In [1]:
# # Notebook Autocontenido — Ranking Naive Trinomial Glicko-2 (Baseline de Referencia)
#
# **Filosofia:** Reedicion metodologica del motor de referencia embebido directamente
# en el entorno de experimentación. Trazabilidad total sin imports externos.
#
# - Motor: Shelopugin-Sirotkin (SS) puro, sin calibración.
# - Probabilidad de empate: Skellam via LightGBM-Poisson (entrenamiento externo).
# - Rating dinámico: iteración secuencial pura, sin optimización.
# - Parámetros: canónicos fijos (μ₀=1500, φ₀=350, τ=0.8, ε=1e-6, draw_correction=0.0).
# - Multi-dataset: procesa lite, full y/o lite_recortado en una sola ejecución.

# ## Celda 1 — Setup, Imports y Selector Multi-Dataset

# ------------------------------------------------------------------
# Montaje de Google Drive (ejecutar solo si estás en Colab)
# ------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------------
# CONFIGURACIÓN GLOBAL
# ------------------------------------------------------------------
BASE_DATA_DIR = "/content/drive/MyDrive/TEG02-2025Arellano/Data"
# BASE_DATA_DIR = r"C:\Users\ruben\Documents\Universidad\TEG\Codigos\AplanarDatos"

# Modo de dataset: "lite" | "full" | "lite_recortado" | "full_lite" | "full_lite_recortado"
DATASET_MODE = "full_lite_recortado"

# ------------------------------------------------------------------
# Imports estándar (notebook 100% autocontenido; sin imports de .py externos)
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
import math
from math import exp, log, sqrt, pi
from dataclasses import dataclass
import lightgbm as lgb
from scipy.special import iv
from sklearn.metrics import mean_absolute_error
import joblib
import json
import os
from scipy.stats import spearmanr
from sklearn.metrics import log_loss

# Para Webscrap
!pip install beautifulsoup4 lxml
!pip install datafc

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from typing import Optional, List, Dict

from datafc import clubelo


# Determinismo
np.random.seed(42)

# ------------------------------------------------------------------
# Selector de datasets activos
# ------------------------------------------------------------------
DATASET_OPTIONS = {
    "lite": ["lite"],
    "full": ["full"],
    "lite_recortado": ["lite_recortado"],
    "full_lite": ["full", "lite"],
    "full_lite_recortado": ["full", "lite", "lite_recortado"],
}

if DATASET_MODE not in DATASET_OPTIONS:
    raise ValueError(
        f"DATASET_MODE inválido: {DATASET_MODE}. "
        f"Opciones válidas: {list(DATASET_OPTIONS.keys())}"
    )

ACTIVE_DATASET_NAMES = DATASET_OPTIONS[DATASET_MODE]
print(f"Modo activo: {DATASET_MODE}")
print(f"Datasets a procesar: {', '.join(ACTIVE_DATASET_NAMES)}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 3.5 MB/s eta 0:00:00
Modo activo: full_lite_recortado
Datasets a procesar: full, lite, lite_recortado


In [2]:
# ## Celda 2 — Motor Glicko-2 Trinomial (Shelopugin-Sirotkin embebido)
#
# Reimplementación estricta del motor SS. No se importa ningún módulo externo.
# - `draw_correction = 0.0` (naive).
# - `advantage` recibido pero anulado en este baseline.
# - Paso 5 (volatilidad) con método de Illinois (regula falsi).

@dataclass
class Rating:
    mu: float = 1500.
    rd: float = 350.
    volatility: float = 0.06


class Glicko2:
    """
    Motor Glicko-2 trinomial basado estrictamente en Shelopugin-Sirotkin.
    """

    def __init__(
        self,
        mu: float = 1500.,
        rd: float = 350.,
        tau: float = 0.8,
        epsilon: float = 0.000001,
        draw_correction: float = 0.0,
    ) -> None:
        self.mu = mu
        self.rd = rd
        self.tau = tau
        self.q = log(10) / 400
        self.q1 = 400 / log(10)
        self.g_constant = 1.5 / (pi ** 2)
        self.epsilon = epsilon
        self.draw_correction = draw_correction

    # ------------------------------------------------------------------
    # Conversión de escala
    # ------------------------------------------------------------------
    def _convert_into_glicko2(self, rating: Rating) -> Rating:
        mu = (rating.mu - self.mu) * self.q
        phi = rating.rd * self.q
        return Rating(mu, phi, rating.volatility)

    def _convert_from_glicko2(self, rating: Rating) -> Rating:
        mu = rating.mu * self.q1 + self.mu
        phi = rating.rd * self.q1
        return Rating(mu, phi, rating.volatility)

    # ------------------------------------------------------------------
    # Funciones auxiliares del algoritmo
    # ------------------------------------------------------------------
    def _g(self, phi: float, phi_opp: float) -> float:
        return 1.0 / sqrt(1.0 + self.g_constant * (phi ** 2 + phi_opp ** 2))

    def _expected_score(
        self,
        mu: float,
        mu_opp: float,
        g: float,
        skellam_draw_probability: float,
    ) -> float:
        delta_mu = g * (mu - mu_opp)
        draw_coefficient = 1.0 + exp(self.draw_correction + skellam_draw_probability)
        exp_delta_mu = exp(delta_mu)

        win_probability = 1.0 / (1.0 + draw_coefficient / exp_delta_mu)
        loss_probability = 1.0 / (1.0 + draw_coefficient * exp_delta_mu)
        tie_probability = 1.0 - win_probability - loss_probability

        return win_probability + 0.5 * tie_probability

    @staticmethod
    def _v(g: float, expected_score: float) -> float:
        denom = g ** 2 * expected_score * (1.0 - expected_score)
        return 1.0 / denom if denom != 0.0 else 1e12

    @staticmethod
    def _rating_improvement(
        v: float, g: float, outcome: float, expected_score: float
    ) -> float:
        return v * g * (outcome - expected_score)

    @staticmethod
    def _f(
        rating_improvement_squared: float,
        v: float,
        phi_squared: float,
        volatility_squared: float,
        x: float,
    ) -> float:
        exp_x = exp(x)
        temp_var = phi_squared + v + exp_x
        first_term = exp_x * (rating_improvement_squared - temp_var) / (
            2.0 * temp_var ** 2
        )
        second_term = x - log(volatility_squared)
        return first_term - second_term

    # ------------------------------------------------------------------
    # Paso 5 — Actualización de volatilidad (Illinois / regula falsi)
    # ------------------------------------------------------------------
    def _update_volatility(
        self,
        rating: Rating,
        rating_opp: Rating,
        outcome: float,
        skellam_draw_probability: float,
    ) -> float:
        mu = rating.mu
        phi = rating.rd
        phi_squared = phi ** 2
        volatility_squared = rating.volatility ** 2
        mu_opp = rating_opp.mu
        phi_opp = rating_opp.rd

        g = self._g(phi, phi_opp)
        expected_score = self._expected_score(mu, mu_opp, g, skellam_draw_probability)
        v = self._v(g, expected_score)
        rating_improvement = self._rating_improvement(v, g, outcome, expected_score)
        rating_improvement_squared = rating_improvement ** 2

        a = log(volatility_squared)

        temp = rating_improvement_squared - phi_squared - v
        if temp > 0.0:
            b = log(temp)
        else:
            k = 1
            while (
                self._f(
                    rating_improvement_squared,
                    v,
                    phi_squared,
                    volatility_squared,
                    a - k * self.tau,
                )
                < 0.0
            ):
                k += 1
            b = a - k * self.tau

        f_a = self._f(
            rating_improvement_squared, v, phi_squared, volatility_squared, a
        )
        f_b = self._f(
            rating_improvement_squared, v, phi_squared, volatility_squared, b
        )

        while abs(b - a) > self.epsilon:
            c = a + (a - b) * f_a / (f_b - f_a)
            f_c = self._f(
                rating_improvement_squared, v, phi_squared, volatility_squared, c
            )
            if f_c * f_b < 0.0:
                a, f_a = b, f_b
            else:
                f_a /= 2.0
            b, f_b = c, f_c

        return exp(a / 2.0)

    # ------------------------------------------------------------------
    # Pasos 6-8 — Actualización de rating y RD
    # ------------------------------------------------------------------
    def _update_rating(
        self,
        rating: Rating,
        rating_opp: Rating,
        outcome: float,
        skellam_draw_probability: float,
    ) -> Rating:
        rating = self._convert_into_glicko2(rating)
        rating_opp = self._convert_into_glicko2(rating_opp)

        mu = rating.mu
        phi = rating.rd
        mu_opp = rating_opp.mu
        phi_opp = rating_opp.rd

        g = self._g(phi, phi_opp)
        expected_score = self._expected_score(mu, mu_opp, g, skellam_draw_probability)
        v = self._v(g, expected_score)

        new_volatility = self._update_volatility(
            rating, rating_opp, outcome, skellam_draw_probability
        )

        squared_phi_star = phi ** 2 + new_volatility ** 2
        new_phi = 1.0 / sqrt((1.0 / squared_phi_star) + (1.0 / v))
        new_mu = mu + (new_phi ** 2) * g * (outcome - expected_score)

        new_rating = Rating(new_mu, new_phi, new_volatility)
        return self._convert_from_glicko2(new_rating)

    @staticmethod
    def _increase_mu(rating: Rating, addition: float) -> Rating:
        return Rating(rating.mu + addition, rating.rd, rating.volatility)

    # ------------------------------------------------------------------
    # Interfaz pública
    # ------------------------------------------------------------------
    def rate(
        self,
        home_rating: Rating,
        away_rating: Rating,
        advantage: float,
        outcome: str,
        skellam_draw_probability: float,
    ) -> tuple[Rating, Rating]:
        """
        Actualiza ratings tras un partido.
        outcome: 'H' | 'D' | 'A'
        """
        increased_home = self._increase_mu(home_rating, advantage)
        decreased_away = self._increase_mu(away_rating, -advantage)

        if outcome == "H":
            new_home = self._update_rating(
                increased_home, decreased_away, 1.0, skellam_draw_probability
            )
            new_away = self._update_rating(
                decreased_away, increased_home, 0.0, skellam_draw_probability
            )
        elif outcome == "D":
            new_home = self._update_rating(
                increased_home, decreased_away, 0.5, skellam_draw_probability
            )
            new_away = self._update_rating(
                decreased_away, increased_home, 0.5, skellam_draw_probability
            )
        else:  # 'A'
            new_home = self._update_rating(
                increased_home, decreased_away, 0.0, skellam_draw_probability
            )
            new_away = self._update_rating(
                decreased_away, increased_home, 1.0, skellam_draw_probability
            )

        updated_home = self._increase_mu(new_home, -advantage)
        updated_away = self._increase_mu(new_away, advantage)

        return updated_home, updated_away


In [3]:
# ## Celda 3 — Bloque Skellam (Entrenamiento por Dataset Activo)
#
# Para cada dataset activo: carga, entrena dos regresores Poisson (LightGBM),
# calcula `draw_prob` via Skellam para todo el dataset, y persiste los modelos.

# %%
# ------------------------------------------------------------------
# Rolling features canónicas (32)
# ------------------------------------------------------------------
rolling_features = [
    "home_goals_r5", "home_Gcon_r5", "home_goals_r10", "home_Gcon_r10",
    "home_goals_r20", "home_Gcon_r20", "home_goals_r30", "home_Gcon_r30",
    "home_goals_home_r5", "home_Gcon_home_r5", "home_goals_home_r10", "home_Gcon_home_r10",
    "home_goals_home_r20", "home_Gcon_home_r20", "home_goals_home_r30", "home_Gcon_home_r30",
    "away_goals_r5", "away_Gcon_r5", "away_goals_r10", "away_Gcon_r10",
    "away_goals_r20", "away_Gcon_r20", "away_goals_r30", "away_Gcon_r30",
    "away_goals_away_r5", "away_Gcon_away_r5", "away_goals_away_r10", "away_Gcon_away_r10",
    "away_goals_away_r20", "away_Gcon_away_r20", "away_goals_away_r30", "away_Gcon_away_r30",
]

# ------------------------------------------------------------------
# Funciones auxiliares
# ------------------------------------------------------------------
def skellam_draw_prob(mu1, mu2):
    """Probabilidad de empate según distribución de Skellam (k=0)."""
    mu1 = np.maximum(mu1, 0.0)
    mu2 = np.maximum(mu2, 0.0)
    x = 2.0 * np.sqrt(mu1 * mu2)
    x = np.clip(x, 0, 700)
    return np.exp(-(mu1 + mu2)) * iv(0, x)


def load_base_dataframes(base_dir: str):
    """Carga los CSV base lite y full."""
    ruta_lite = os.path.join(base_dir, "matches_final_model_clean2.csv")
    ruta_full = os.path.join(base_dir, "matches_final_model_full2.csv")
    if not os.path.exists(ruta_lite):
        raise FileNotFoundError(f"No se encontró: {ruta_lite}")
    if not os.path.exists(ruta_full):
        raise FileNotFoundError(f"No se encontró: {ruta_full}")
    return pd.read_csv(ruta_lite), pd.read_csv(ruta_full)


def build_lite_recortado(df_lite_source, df_full_source):
    if (
        "match_id_new" not in df_lite_source.columns
        or "match_id_new" not in df_full_source.columns
    ):
        raise ValueError("Se requiere 'match_id_new' en ambos datasets para construir lite_recortado")
    common_ids = pd.Index(df_full_source["match_id_new"].dropna().unique())
    return df_lite_source.loc[df_lite_source["match_id_new"].isin(common_ids)].copy()


def get_available_rolling_features(df):
    cols = [c for c in rolling_features if c in df.columns]
    missing = [c for c in rolling_features if c not in df.columns]
    if missing:
        print(f"  Advertencia: faltan {len(missing)} rolling features: {missing}")
    if not cols:
        raise ValueError("No hay rolling features disponibles.")
    return cols


# ------------------------------------------------------------------
# Carga base
# ------------------------------------------------------------------
df_lite_raw, df_full_raw = load_base_dataframes(BASE_DATA_DIR)

# Construir lite_recortado si es necesario
if "lite_recortado" in ACTIVE_DATASET_NAMES:
    df_lite_recortado_raw = build_lite_recortado(df_lite_raw, df_full_raw)
    print(f"/lite_recortado/ construido: {len(df_lite_recortado_raw)} filas")
else:
    df_lite_recortado_raw = None

# Registro de datasets
DATASET_REGISTRY = {
    "lite": df_lite_raw,
    "full": df_full_raw,
    "lite_recortado": df_lite_recortado_raw,
}

# ------------------------------------------------------------------
# Entrenamiento Skellam por dataset activo
# ------------------------------------------------------------------
skellam_results = {}  # dataset_name -> {"df": df_con_draw, "model_home": ..., "model_away": ...}

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*60}")
    print(f" PROCESANDO SKELLAM: {dataset_name.upper()}")
    print(f"{'='*60}")

    df_src = DATASET_REGISTRY[dataset_name].copy()

    # Verificar columnas mínimas
    required = ["home_team", "away_team", "home_goals", "away_goals", "match_date_from_id"]
    for col in required:
        if col not in df_src.columns:
            raise ValueError(f"[{dataset_name}] Falta columna obligatoria: {col}")

    # Orden cronológico
    df_src = df_src.sort_values("match_date_from_id").reset_index(drop=True)

    # Features disponibles
    available_rolling = get_available_rolling_features(df_src)

    # Split 70/30 temporal
    n_train = int(len(df_src) * 0.7)
    train_df = df_src.iloc[:n_train].copy()
    test_df = df_src.iloc[n_train:].copy()

    print(f"  Total: {len(df_src)} | Train: {len(train_df)} | Test: {len(test_df)}")

    # Entrenar Poisson
    X_train = train_df[available_rolling]
    y_home_train = train_df["home_goals"]
    y_away_train = train_df["away_goals"]

    model_home = lgb.LGBMRegressor(
        objective="poisson", metric="mae",
        n_estimators=200, learning_rate=0.05, max_depth=5,
        random_state=42, verbose=-1,
    )
    model_away = lgb.LGBMRegressor(
        objective="poisson", metric="mae",
        n_estimators=200, learning_rate=0.05, max_depth=5,
        random_state=42, verbose=-1,
    )

    model_home.fit(X_train, y_home_train)
    model_away.fit(X_train, y_away_train)

    # Predecir sobre todo el dataset
    X_full = df_src[available_rolling]
    df_src["mu_home"] = model_home.predict(X_full)
    df_src["mu_away"] = model_away.predict(X_full)

    mu_h = np.maximum(df_src["mu_home"].values, 0.0)
    mu_a = np.maximum(df_src["mu_away"].values, 0.0)
    df_src["draw_prob"] = skellam_draw_prob(mu_h, mu_a)

    # Persistir modelos
    joblib.dump(model_home, f"skellam_naive_{dataset_name}_home_model.pkl")
    joblib.dump(model_away, f"skellam_naive_{dataset_name}_away_model.pkl")
    print(f"  Modelos guardados: skellam_naive_{dataset_name}_home/away_model.pkl")

    # Guardar en registro
    skellam_results[dataset_name] = {
        "df": df_src,
        "model_home": model_home,
        "model_away": model_away,
    }

    print(f"  draw_prob stats:")
    print(df_src["draw_prob"].describe())



/lite_recortado/ construido: 26960 filas

 PROCESANDO SKELLAM: FULL
  Total: 26960 | Train: 18872 | Test: 8088
  Modelos guardados: skellam_naive_full_home/away_model.pkl
  draw_prob stats:
count    26960.000000
mean         0.252742
std          0.032094
min          0.037238
25%          0.237789
50%          0.255364
75%          0.274453
max          0.359825
Name: draw_prob, dtype: float64

 PROCESANDO SKELLAM: LITE
  Total: 37712 | Train: 26398 | Test: 11314
  Modelos guardados: skellam_naive_lite_home/away_model.pkl
  draw_prob stats:
count    37712.000000
mean         0.255736
std          0.032731
min          0.036609
25%          0.239552
50%          0.257894
75%          0.277005
max          0.383294
Name: draw_prob, dtype: float64

 PROCESANDO SKELLAM: LITE_RECORTADO
  Total: 26960 | Train: 18872 | Test: 8088
  Modelos guardados: skellam_naive_lite_recortado_home/away_model.pkl
  draw_prob stats:
count    26960.000000
mean         0.252742
std          0.032094
min      

In [4]:
# ## Celda 4 — Loop Secuencial Naive (por Dataset Activo)
#
# Para cada dataset activo: instancia un motor limpio, itera cronológicamente
# partido a partido, y acumula el historial de ratings.

# Resultados globales
glicko_outputs = {}  # dataset_name -> {"ratings": dict, "history": DataFrame}

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*60}")
    print(f" EJECUTANDO GLICKO-2: {dataset_name.upper()}")
    print(f"{'='*60}")

    df = skellam_results[dataset_name]["df"].copy()
    df = df.sort_values("match_date_from_id").reset_index(drop=True)

    # Motor limpio por dataset (misma semilla conceptual)
    glicko = Glicko2(mu=1500.0, rd=350.0, tau=0.8, epsilon=0.000001, draw_correction=0.0)
    ratings = {}

    def get_rating(team: str) -> Rating:
        if team not in ratings:
            ratings[team] = Rating()
        return ratings[team]

    history = []

    for idx, row in df.iterrows():
        home_team = row["home_team"]
        away_team = row["away_team"]

        home_rating = get_rating(home_team)
        away_rating = get_rating(away_team)

        if row["home_goals"] > row["away_goals"]:
            outcome = "H"
        elif row["home_goals"] < row["away_goals"]:
            outcome = "A"
        else:
            outcome = "D"

        skellam_draw_probability = row["draw_prob"]

        new_home, new_away = glicko.rate(
            home_rating, away_rating,
            advantage=0.0,
            outcome=outcome,
            skellam_draw_probability=skellam_draw_probability,
        )

        ratings[home_team] = new_home
        ratings[away_team] = new_away

        history.append({
            "match_id": row.get("match_id_new", idx),
            "match_date": row["match_date_from_id"],
            "home_team": home_team,
            "away_team": away_team,
            "home_mu": new_home.mu,
            "home_rd": new_home.rd,
            "home_volatility": new_home.volatility,
            "away_mu": new_away.mu,
            "away_rd": new_away.rd,
            "away_volatility": new_away.volatility,
            "outcome": outcome,
            "draw_prob": skellam_draw_probability,
        })

    df_history = pd.DataFrame(history)
    glicko_outputs[dataset_name] = {
        "ratings": ratings,
        "history": df_history,
    }

    print(f"  Partidos procesados : {len(df_history)}")
    print(f"  Equipos únicos      : {len(ratings)}")


 EJECUTANDO GLICKO-2: FULL


  Partidos procesados : 26960
  Equipos únicos      : 336

 EJECUTANDO GLICKO-2: LITE
  Partidos procesados : 37712
  Equipos únicos      : 414

 EJECUTANDO GLICKO-2: LITE_RECORTADO
  Partidos procesados : 26960
  Equipos únicos      : 336


In [5]:
# ## Celda 5 — Exportación y Resumen (por Dataset Activo)

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*70}")
    print(f" EXPORTACIÓN Y RESUMEN: {dataset_name.upper()}")
    print(f"{'='*70}")

    ratings = glicko_outputs[dataset_name]["ratings"]
    df_history = glicko_outputs[dataset_name]["history"]

    # Conteo de partidos
    home_counts = df_history["home_team"].value_counts()
    away_counts = df_history["away_team"].value_counts()
    n_partidos = home_counts.add(away_counts, fill_value=0).astype(int)

    # Tabla final
    records = []
    for team, rating in ratings.items():
        records.append({
            "team": team,
            "mu": rating.mu,
            "rd": rating.rd,
            "volatility": rating.volatility,
            "n_partidos": n_partidos.get(team, 0),
        })
    df_ratings_final = pd.DataFrame(records)

    # Exportar
    out_final = os.path.join(BASE_DATA_DIR, f"ratings_final_naive_trinomial_{dataset_name}.csv")
    out_history = os.path.join(BASE_DATA_DIR, f"ratings_history_naive_trinomial_{dataset_name}.csv")

    df_ratings_final.to_csv(out_final, index=False)
    df_history.to_csv(out_history, index=False)

    print(f"  CSV exportado: {out_final}")
    print(f"  CSV exportado: {out_history}")

    # Resumen
    top20 = df_ratings_final.sort_values("mu", ascending=False).head(20)
    print("\n  >>> Top 20 equipos por rating mu descendente:")
    print(top20[["team", "mu", "rd", "volatility", "n_partidos"]].to_string(index=False))

    mu_range = df_ratings_final["mu"].max() - df_ratings_final["mu"].min()
    print(f"\n  >>> Rango de ratings (max(mu) − min(mu)): {mu_range:.2f}")
    print(f"  >>> Número total de equipos unicos      : {df_ratings_final['team'].nunique()}")

    corr, pval = spearmanr(df_ratings_final["mu"], df_ratings_final["n_partidos"])
    print(f"  >>> Correlación Spearman (mu vs n_partidos): {corr:.4f} (p = {pval:.4f})")

    print(f"\n  >>> Estadísticas descriptivas de draw_prob:")
    print(skellam_results[dataset_name]["df"]["draw_prob"].describe())



 EXPORTACIÓN Y RESUMEN: FULL
  CSV exportado: /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_final_naive_trinomial_full.csv
  CSV exportado: /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_history_naive_trinomial_full.csv

  >>> Top 20 equipos por rating mu descendente:
      team          mu        rd  volatility  n_partidos
       PSG 1833.971468 64.078015    0.059504         270
RealMadrid 1795.323728 64.612293    0.059334         289
   AlHilal 1779.653205 68.013447    0.059513         197
    Bayern 1777.745753 64.831162    0.059679         250
 Liverpool 1772.714447 65.371109    0.059578         266
Olympiacos 1765.589180 63.442262    0.059400         242
InterMilan 1756.126151 64.136947    0.059556         177
 Barcelona 1747.436279 64.118198    0.059384         264
   ManCity 1743.448952 64.526443    0.059560         276
   Arsenal 1740.492345 63.983533    0.059306         252
 AtlMadrid 1728.926191 63.721203    0.059422         264
 AlIttihad 1714.776090 64.360

In [6]:
# ## Celda 6 — Sanity Check (por Dataset Activo)

print("=" * 70)
print(" SANITY CHECK MULTI-DATASET")
print("=" * 70)


def decompose_probabilities(home_mu, home_rd, away_mu, away_rd, draw_prob):
    """Reconstruye la tríada (W, T, L) usando la misma lógica del motor SS."""
    q = log(10) / 400
    g_c = 1.5 / (pi ** 2)

    mu = (home_mu - 1500.0) * q
    phi = home_rd * q
    mu_opp = (away_mu - 1500.0) * q
    phi_opp = away_rd * q

    g = 1.0 / sqrt(1.0 + g_c * (phi ** 2 + phi_opp ** 2))
    delta_mu = g * (mu - mu_opp)

    draw_coefficient = 1.0 + exp(0.0 + draw_prob)
    exp_delta_mu = exp(delta_mu)

    win = 1.0 / (1.0 + draw_coefficient / exp_delta_mu)
    loss = 1.0 / (1.0 + draw_coefficient * exp_delta_mu)
    tie = 1.0 - win - loss
    return win, tie, loss


for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n--- Dataset: {dataset_name} ---")

    out_final = os.path.join(BASE_DATA_DIR, f"ratings_final_naive_trinomial_{dataset_name}.csv")
    out_history = os.path.join(BASE_DATA_DIR, f"ratings_history_naive_trinomial_{dataset_name}.csv")

    # 1. Existencia CSVs
    for fname in [out_final, out_history]:
        exists = os.path.exists(fname)
        size = os.path.getsize(fname) if exists else 0
        status = "OK" if exists and size > 0 else "FALLO"
        print(f"  [{status}] {fname}")

    df_ratings_final = pd.read_csv(out_final)
    df_history = pd.read_csv(out_history)

    # 2. NaN en finales
    nan_final = df_ratings_final[["team", "mu", "rd", "volatility"]].isna().sum().sum()
    print(f"  [NaN finales: {nan_final}] {'OK' if nan_final == 0 else 'FALLO'}")

    # 3. NaN en historial
    nan_hist = df_history[[
        "match_id", "match_date", "home_team", "away_team",
        "home_mu", "home_rd", "home_volatility",
        "away_mu", "away_rd", "away_volatility",
        "outcome", "draw_prob",
    ]].isna().sum().sum()
    print(f"  [NaN historial: {nan_hist}] {'OK' if nan_hist == 0 else 'FALLO'}")

    # 4. Suma de probabilidades en muestra aleatoria
    np.random.seed(42)
    sample = df_history.sample(n=min(5, len(df_history))).reset_index(drop=True)

    print(f"  Suma de probabilidades (muestra n=5):")
    for i, row in sample.iterrows():
        w, t, l = decompose_probabilities(
            row["home_mu"], row["home_rd"],
            row["away_mu"], row["away_rd"],
            row["draw_prob"],
        )
        total = w + t + l
        ok = abs(total - 1.0) < 1e-6
        print(f"    {i+1}. {row['home_team']} vs {row['away_team']} | "
              f"Sum={total:.10f} | {'OK' if ok else 'DESVIO'}")

print("\n[SANITY CHECK COMPLETADO PARA TODOS LOS DATASETS ACTIVOS]")

 SANITY CHECK MULTI-DATASET

--- Dataset: full ---
  [OK] /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_final_naive_trinomial_full.csv
  [OK] /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_history_naive_trinomial_full.csv
  [NaN finales: 0] OK
  [NaN historial: 0] OK
  Suma de probabilidades (muestra n=5):
    1. AlHilal vs AlEttifaq | Sum=1.0000000000 | OK
    2. Freiburg vs Hertha | Sum=1.0000000000 | OK
    3. AlRaed vs Abha | Sum=1.0000000000 | OK
    4. Brighton vs SheffieldUtd | Sum=1.0000000000 | OK
    5. NERevolution vs DCUtd | Sum=1.0000000000 | OK

--- Dataset: lite ---
  [OK] /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_final_naive_trinomial_lite.csv
  [OK] /content/drive/MyDrive/TEG02-2025Arellano/Data/ratings_history_naive_trinomial_lite.csv
  [NaN finales: 0] OK
  [NaN historial: 0] OK
  Suma de probabilidades (muestra n=5):
    1. Austin vs AtlantaUtd | Sum=1.0000000000 | OK
    2. Damac vs AlTaawoun | Sum=1.0000000000 | OK
    3. Celta vs Atl

In [7]:
# %% Celda 7 — POST-PROCESAMIENTO: MÉTRICAS COMPLETAS DEL NAIVE (corregida)

# Función vectorizada para descomponer probabilidades (usando np en lugar de math)
def decompose_probabilities(home_mu, home_rd, away_mu, away_rd, draw_prob):
    q = np.log(10) / 400
    g_c = 1.5 / (np.pi ** 2)
    # Conversión a escala Glicko-2
    mu = (home_mu - 1500.0) * q
    phi = home_rd * q
    mu_opp = (away_mu - 1500.0) * q
    phi_opp = away_rd * q
    g = 1.0 / np.sqrt(1.0 + g_c * (phi ** 2 + phi_opp ** 2))
    delta_mu = g * (mu - mu_opp)
    draw_coefficient = 1.0 + np.exp(0.0 + draw_prob)
    exp_delta_mu = np.exp(delta_mu)
    win = 1.0 / (1.0 + draw_coefficient / exp_delta_mu)
    loss = 1.0 / (1.0 + draw_coefficient * exp_delta_mu)
    tie = 1.0 - win - loss
    return win, tie, loss

# Mapeo de resultados a índices para multiclase
outcome_to_idx = {'H': 0, 'D': 1, 'A': 2}

print("=" * 70)
print(" CÁLCULO DE MÉTRICAS COMPLETAS PARA EL MOTOR NAIVE (basado en CSVs generados)")
print("=" * 70)

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*60}")
    print(f" DATASET: {dataset_name.upper()}")
    print(f"{'='*60}")
    
    # 1. Cargar historial (contiene las probabilidades base y los outcomes)
    hist_path = os.path.join(BASE_DATA_DIR, f"ratings_history_naive_trinomial_{dataset_name}.csv")
    df_hist = pd.read_csv(hist_path)
    
    # 2. Reconstruir el vector de probabilidades (P_H, P_D, P_A) para cada partido
    P_H, P_D, P_A = decompose_probabilities(
        df_hist["home_mu"].values,
        df_hist["home_rd"].values,
        df_hist["away_mu"].values,
        df_hist["away_rd"].values,
        df_hist["draw_prob"].values
    )
    
    # 3. Recortar para evitar log(0) en la log-loss
    eps = 1e-15
    P_H = np.clip(P_H, eps, 1 - eps)
    P_D = np.clip(P_D, eps, 1 - eps)
    P_A = np.clip(P_A, eps, 1 - eps)
    probas = np.column_stack([P_H, P_D, P_A])  # shape (N, 3)
    
    # Verificar que las probabilidades suman 1 (dentro de tolerancia)
    sum_probas = np.sum(probas, axis=1)
    if not np.allclose(sum_probas, 1.0, atol=1e-6):
        print("  Advertencia: algunas filas no suman 1. Re-normalizando...")
        probas = probas / sum_probas[:, np.newaxis]
        # Reasignar P_H, P_D, P_A por si se usan más adelante
        P_H, P_D, P_A = probas[:, 0], probas[:, 1], probas[:, 2]
    
    # 4. Target multiclase: índices 0,1,2
    y_true_indices = df_hist["outcome"].map(outcome_to_idx).values
    
    # 5. Log-Loss CATEGÓRICA (H/D/A) con y_true como índices
    ll_cat = log_loss(y_true_indices, probas)
    
    # 6. Log-Loss BINARIA (H vs no H)
    y_true_bin = (df_hist["outcome"] == "H").astype(int)
    ll_bin = log_loss(y_true_bin, P_H)
    
    # 7. Cargar ratings finales para Spearman y rango
    final_path =  os.path.join(BASE_DATA_DIR, f"ratings_final_naive_trinomial_{dataset_name}.csv")
    df_final = pd.read_csv(final_path)
    
    # Contar partidos por equipo (si no está en el CSV final)
    home_counts = df_hist["home_team"].value_counts()
    away_counts = df_hist["away_team"].value_counts()
    n_partidos = home_counts.add(away_counts, fill_value=0).astype(int)
    df_final["n_partidos"] = df_final["team"].map(n_partidos).fillna(0).astype(int)
    
    rango = df_final["mu"].max() - df_final["mu"].min()
    n_equipos = df_final["team"].nunique()
    corr, pval = spearmanr(df_final["mu"], df_final["n_partidos"])
    
    # 8. Imprimir resultados
    print(f"  >>> Log-Loss CATEGÓRICA (H/D/A) : {ll_cat:.6f}")
    print(f"  >>> Log-Loss BINARIA (H vs no H): {ll_bin:.6f}")
    print(f"  >>> Rango de ratings (max-min)   : {rango:.2f}")
    print(f"  >>> Equipos únicos               : {n_equipos}")
    print(f"  >>> Spearman (mu vs n_partidos)   : {corr:.4f} (p = {pval:.4f})")

 CÁLCULO DE MÉTRICAS COMPLETAS PARA EL MOTOR NAIVE (basado en CSVs generados)

 DATASET: FULL
  >>> Log-Loss CATEGÓRICA (H/D/A) : 0.984692
  >>> Log-Loss BINARIA (H vs no H): 0.620071
  >>> Rango de ratings (max-min)   : 614.55
  >>> Equipos únicos               : 336
  >>> Spearman (mu vs n_partidos)   : 0.5976 (p = 0.0000)

 DATASET: LITE
  >>> Log-Loss CATEGÓRICA (H/D/A) : 0.985102
  >>> Log-Loss BINARIA (H vs no H): 0.620036
  >>> Rango de ratings (max-min)   : 731.12
  >>> Equipos únicos               : 414
  >>> Spearman (mu vs n_partidos)   : 0.5775 (p = 0.0000)

 DATASET: LITE_RECORTADO
  >>> Log-Loss CATEGÓRICA (H/D/A) : 0.984692
  >>> Log-Loss BINARIA (H vs no H): 0.620071
  >>> Rango de ratings (max-min)   : 614.55
  >>> Equipos únicos               : 336
  >>> Spearman (mu vs n_partidos)   : 0.5976 (p = 0.0000)


In [12]:
# %% Celda 8 — COMPARACIÓN NAIVE vs G2Av1513 (por dataset)
# =============================================================================
# Compara los ratings finales y rankings entre el motor Naive (baseline)
# y el motor calibrado G2Av1513 (cuerpo del trabajo).
# =============================================================================

# Directorio donde están los CSVs (el mismo donde se guardaron los Naive)
COMPARE_DIR = BASE_DATA_DIR  

# Prefijos de archivos
NAIVE_PREFIX = "ratings_final_naive_trinomial"
G2_PREFIX = "glicko_ratings_final"

# Sufijos según dataset
DATASET_SUFFIX = {
    "full": "full",
    "lite": "lite",
    "lite_recortado": "lite_recortado",
}

# Diccionario para almacenar resultados de comparación
comparison_results = {}

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*70}")
    print(f" COMPARACIÓN NAIVE vs G2Av1513 — {dataset_name.upper()}")
    print(f"{'='*70}")
    
    suffix = DATASET_SUFFIX[dataset_name]
    
    # 1. Construir rutas
    naive_file = os.path.join(COMPARE_DIR, f"{NAIVE_PREFIX}_{suffix}.csv")
    g2_file = os.path.join(COMPARE_DIR, f"{G2_PREFIX}_{suffix}.csv")
    
    # 2. Verificar existencia
    if not os.path.exists(naive_file):
        print(f"  ERROR: No se encuentra {naive_file}")
        continue
    if not os.path.exists(g2_file):
        print(f"  ERROR: No se encuentra {g2_file}")
        continue
    
    # 3. Cargar CSVs
    df_naive = pd.read_csv(naive_file)
    df_g2 = pd.read_csv(g2_file)
    
    # 4. Normalizar nombre de columna de equipo (si es necesario)
    # Asumimos que la columna se llama 'team'. Si se llama 'home_team' o similar, renombrar.
    if 'team' not in df_naive.columns:
        # Buscar columna que contenga 'team' o 'Team'
        possible = [c for c in df_naive.columns if 'team' in c.lower()]
        if possible:
            df_naive.rename(columns={possible[0]: 'team'}, inplace=True)
        else:
            print("  ERROR: No se encontro columna de equipo en df_naive")
            continue
    if 'team' not in df_g2.columns:
        possible = [c for c in df_g2.columns if 'team' in c.lower()]
        if possible:
            df_g2.rename(columns={possible[0]: 'team'}, inplace=True)
        else:
            print("  ERROR: No se encontro columna de equipo en df_g2")
            continue
    
    # 5. Unir por equipo (inner join para solo equipos en ambos)
    df_merged = pd.merge(df_naive, df_g2, on='team', suffixes=('_naive', '_g2'))
    
    # 6. Calcular rankings (orden descendente por mu)
    df_naive_sorted = df_naive.sort_values('mu', ascending=False).reset_index(drop=True)
    df_g2_sorted = df_g2.sort_values('mu', ascending=False).reset_index(drop=True)
    df_naive_sorted['rank_naive'] = df_naive_sorted.index + 1
    df_g2_sorted['rank_g2'] = df_g2_sorted.index + 1
    
    # Unir rankings al merged
    df_merged = pd.merge(df_merged, df_naive_sorted[['team', 'rank_naive']], on='team', how='left')
    df_merged = pd.merge(df_merged, df_g2_sorted[['team', 'rank_g2']], on='team', how='left')
    
    # 7. Calcular diferencias
    df_merged['mu_diff'] = df_merged['mu_naive'] - df_merged['mu_g2']
    df_merged['mu_diff_abs'] = np.abs(df_merged['mu_diff'])
    df_merged['rank_diff'] = df_merged['rank_naive'] - df_merged['rank_g2']
    df_merged['rank_diff_abs'] = np.abs(df_merged['rank_diff'])
    
    # 8. Estadísticas globales
    n_equipos = len(df_merged)
    avg_mu_diff = df_merged['mu_diff_abs'].mean()
    std_mu_diff = df_merged['mu_diff_abs'].std()
    max_mu_diff = df_merged['mu_diff_abs'].max()
    avg_rank_diff = df_merged['rank_diff_abs'].mean()
    std_rank_diff = df_merged['rank_diff_abs'].std()
    max_rank_diff = df_merged['rank_diff_abs'].max()
    
    # Correlación de Spearman entre los ratings (no rankings)
    corr, pval = spearmanr(df_merged['mu_naive'], df_merged['mu_g2'])
    
    print(f"\n  >>> Equipos en comun: {n_equipos}")
    print(f"  >>> Diferencia absoluta de mu: promedio={avg_mu_diff:.2f}, std={std_mu_diff:.2f}, max={max_mu_diff:.2f}")
    print(f"  >>> Diferencia absoluta de ranking: promedio={avg_rank_diff:.2f}, std={std_rank_diff:.2f}, max={max_rank_diff:.2f}")
    print(f"  >>> Correlación Spearman (mu_naive vs mu_g2): {corr:.4f} (p={pval:.4f})")
    
    # 9. Top 5 equipos con mayor diferencia de rating
    top5_mu = df_merged.nlargest(5, 'mu_diff_abs')[
        ['team', 'mu_naive', 'mu_g2', 'mu_diff_abs', 'rank_naive', 'rank_g2']
    ]
    print("\n  >>> Top 5 equipos con mayor diferencia absoluta de mu:")
    print(top5_mu.to_string(index=False))
    
    # 10. Top 5 equipos con mayor diferencia de ranking
    top5_rank = df_merged.nlargest(5, 'rank_diff_abs')[
        ['team', 'rank_naive', 'rank_g2', 'rank_diff_abs', 'mu_naive', 'mu_g2']
    ]
    print("\n  >>> Top 5 equipos con mayor diferencia absoluta de ranking:")
    print(top5_rank.to_string(index=False))
    
    # Guardar resultados
    comparison_results[dataset_name] = {
        'df_merged': df_merged,
        'n_equipos': n_equipos,
        'avg_mu_diff': avg_mu_diff,
        'std_mu_diff': std_mu_diff,
        'max_mu_diff': max_mu_diff,
        'avg_rank_diff': avg_rank_diff,
        'std_rank_diff': std_rank_diff,
        'max_rank_diff': max_rank_diff,
        'spearman_corr': corr,
        'spearman_p': pval,
        'top5_mu': top5_mu,
        'top5_rank': top5_rank,
    }

# 11. Resumen global
print("\n" + "="*70)
print(" RESUMEN COMPARATIVO GLOBAL (todos los datasets activos)")
print("="*70)
for ds, res in comparison_results.items():
    print(f"\n{ds.upper()}:")
    print(f"  Avg |mu_naive - mu_g2|: {res['avg_mu_diff']:.2f}  (std={res['std_mu_diff']:.2f})")
    print(f"  Avg |rank_naive - rank_g2|: {res['avg_rank_diff']:.2f}  (std={res['std_rank_diff']:.2f})")
    print(f"  Spearman: {res['spearman_corr']:.4f} (p={res['spearman_p']:.4f})")
    print(f"  Equipos en comun: {res['n_equipos']}")

# Usamos Lite porque tiene más equipos.
df_raw = pd.read_csv(os.path.join(BASE_DATA_DIR, "matches_final_model_clean2.csv"))

# Construir mapeo: 'team' -> 'Competition'

EXCLUDED_COMPETITIONS = ['CONCACAFChampionsCup','CONCACAFChampionsLeague', 
'ChampionsLeague', 'CopaLibertadores', 'CopaSudamericana', 
'EuropaConferenceLeague', 'EuropaLeague', 'FIFAClubWorldCup', 
'AFCChampionsLeagueElite','SuperligaArgentina',
]
df_filter = df_raw[~df_raw['Competition'].isin(EXCLUDED_COMPETITIONS)]
team_league_map = pd.concat([
    df_filter[['home_team', 'Competition']].rename(columns={'home_team': 'team'}),
    df_filter[['away_team', 'Competition']].rename(columns={'away_team': 'team'})
]).drop_duplicates(subset=['team']).set_index('team')['Competition'].to_dict()

# Diccionarios para almacenar los resultados
top10_results = {}
top_league_results = {}

for dataset_name, res in comparison_results.items():
    print(f"\n{'='*70}")
    print(f" TOP 10 Y TOP POR LIGA — {dataset_name.upper()}")
    print(f"{'='*70}")
    
    df_merged = res['df_merged'].copy()
    df_merged['league'] = df_merged['team'].map(team_league_map)
    
    n_mapped = df_merged['league'].notna().sum()
    n_total = len(df_merged)
    print(f"  Equipos mapeados a liga: {n_mapped}/{n_total}")
    
    # --- 1. TOP 10 GLOBAL ---
    top10 = df_merged.nlargest(10, 'mu_diff_abs')[
        ['team', 'league', 'mu_naive', 'mu_g2', 'mu_diff_abs', 'rank_naive', 'rank_g2', 'rank_diff_abs']
    ].reset_index(drop=True)
    top10_results[dataset_name] = top10
    
    print("\n  >>> TOP 10 GLOBAL (mayor |Δμ|):")
    print(top10.to_string(index=False))
    
    # --- 2. TOP POR LIGA ---
    top_league = (
        df_merged.loc[df_merged.groupby('league')['mu_diff_abs'].idxmax()]
        .sort_values('mu_diff_abs', ascending=False)
        .head(10)
        [['league', 'team', 'mu_naive', 'mu_g2', 'mu_diff_abs', 'rank_naive', 'rank_g2']]
        .reset_index(drop=True)
    )
    top_league_results[dataset_name] = top_league
    
    print("\n  >>> TOP 10 LIGAS (mayor |Δμ| del equipo más discrepante de cada liga):")
    print(top_league.to_string(index=False))

for dataset_name in ACTIVE_DATASET_NAMES:
    print(f"\n{'='*70}")
    print(f" TOP 10 — {dataset_name.upper()}")
    print(f"{'='*70}")
    
    suffix = {
        "full": "full",
        "lite": "lite",
        "lite_recortado": "lite_recortado"
    }[dataset_name]
    
    # 1. Cargar CSVs
    naive_file = os.path.join(BASE_DATA_DIR, f"{NAIVE_PREFIX}_{suffix}.csv")
    g2_file = os.path.join(BASE_DATA_DIR, f"{G2_PREFIX}_{suffix}.csv")
    
    df_naive = pd.read_csv(naive_file)
    df_g2 = pd.read_csv(g2_file)
    
    # 2. Ordenar y tomar Top 10
    top_naive = df_naive.nlargest(10, 'mu')[['team', 'mu', 'rd', 'volatility']].reset_index(drop=True)
    top_g2 = df_g2.nlargest(10, 'mu')[['team', 'mu', 'rd', 'volatility']].reset_index(drop=True)
    
    # 3. Imprimir
    print("\n  >>> TOP 10 NAIVE (mayor mu):")
    print(top_naive.to_string(index=False))
    
    print("\n  >>> TOP 10 G2 (mayor mu):")
    print(top_g2.to_string(index=False))


 COMPARACIÓN NAIVE vs G2Av1513 — FULL

  >>> Equipos en comun: 336
  >>> Diferencia absoluta de mu: promedio=40.91, std=27.31, max=233.22
  >>> Diferencia absoluta de ranking: promedio=23.24, std=23.65, max=201.00
  >>> Correlación Spearman (mu_naive vs mu_g2): 0.9416 (p=0.0000)

  >>> Top 5 equipos con mayor diferencia absoluta de mu:
         team    mu_naive       mu_g2  mu_diff_abs  rank_naive  rank_g2
          CSA 1232.969284 1466.184865   233.215580         334      133
     Veracruz 1219.421730 1372.775367   153.353637         336      240
      Toronto 1385.835463 1270.220809   115.614654         264      327
  Southampton 1273.666395 1172.034415   101.631980         328      336
SJEarthquakes 1419.100387 1320.344479    98.755908         220      297

  >>> Top 5 equipos con mayor diferencia absoluta de ranking:
       team  rank_naive  rank_g2  rank_diff_abs    mu_naive       mu_g2
        CSA         334      133            201 1232.969284 1466.184865
     Xanthi         29

In [ ]:
# ## Celda 9 - Definiciones para parseo y Webscrap

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
TIMEOUT = 15

def fetch_html(url: str, headers: dict = HEADERS, timeout: int = TIMEOUT) -> Optional[str]:
    """Obtiene el HTML de una URL con manejo de errores."""
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        # Detectar encoding (algunas páginas usan UTF-8, otras ISO-8859-1)
        if response.encoding is None:
            response.encoding = 'utf-8'
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error al obtener {url}: {e}")
        return None

# ============================================================
# 1. PARSEO UEFA (kassiesa.net)
# ============================================================
def parse_uefa(html: str) -> pd.DataFrame:
    """
    Extrae la tabla de clubes del ranking UEFA.
    Nota: adaptar selectores según inspección real de la página.
    """
    soup = BeautifulSoup(html, 'html.parser')
    
    # Buscar la tabla principal - puede tener un id o clase específica
    # Ejemplo hipotético (ajustar después de inspeccionar):
    table = soup.find('table', {'class': 'ranking'})  # o 'id': 'clubRanking'
    if not table:
        # Fallback: buscar cualquier tabla con más de 10 filas
        tables = soup.find_all('table')
        for t in tables:
            rows = t.find_all('tr')
            if len(rows) > 10:
                table = t
                break
    
    if not table:
        raise ValueError("No se encontró la tabla UEFA")
    
    # Extraer filas
    rows = []
    for tr in table.find_all('tr'):
        cols = tr.find_all(['td', 'th'])
        row_data = [col.get_text(strip=True) for col in cols]
        if row_data:
            rows.append(row_data)
    
    # Asumimos que la primera fila es el encabezado
    if not rows:
        raise ValueError("La tabla UEFA está vacía")
    
    df = pd.DataFrame(rows[1:], columns=rows[0])
    return df

# ============================================================
# 2. PARSEO AFC (footyrankings.github.io)
# ============================================================
def parse_afc(html: str) -> pd.DataFrame:
    """
    Extrae datos del ranking AFC seleccionando la tabla HTML más grande
    que contenga datos numéricos en su primera columna.
    """
    soup = BeautifulSoup(html, 'html.parser')
    
    # 1. Encontrar TODAS las tablas
    all_tables = soup.find_all('table')
    if not all_tables:
        raise ValueError("No se encontraron tablas HTML en la página AFC")
    
    # 2. Filtrar tablas con al menos 3 filas y que tengan números en la 1ra columna
    valid_tables = []
    for table in all_tables:
        rows = table.find_all('tr')
        if len(rows) < 3:
            continue
        
        # Revisar las primeras 5 filas (para evitar falsos negativos)
        has_numeric = False
        for row in rows[1:6]:  # Saltamos posible header
            cols = row.find_all(['td', 'th'])
            if cols:
                first_cell = cols[0].get_text(strip=True).replace(',', '').replace('.', '', 1)
                if first_cell.isdigit():
                    has_numeric = True
                    break
        if has_numeric:
            valid_tables.append(table)
    
    if not valid_tables:
        raise ValueError("No se encontró ninguna tabla con datos numéricos en AFC")
    
    # 3. CRITERIO CLAVE: Elegir la tabla con MÁS FILAS (esa es la del ranking)
    target_table = max(valid_tables, key=lambda t: len(t.find_all('tr')))
    
    # 4. Extraer datos
    rows = target_table.find_all('tr')
    data_rows = []
    header = None
    
    for tr in rows:
        cols = tr.find_all(['td', 'th'])
        row_text = [col.get_text(strip=True) for col in cols]
        
        if not row_text or all(c == '' for c in row_text):
            continue
        
        # Detectar encabezado: si NO empieza con número
        first_val = row_text[0].replace(',', '').replace('.', '', 1)
        if not first_val.isdigit():
            if header is None:
                header = row_text
        else:
            data_rows.append(row_text)
    
    if not data_rows:
        raise ValueError("No se encontraron filas de datos numéricos")
    
    # 5. Ajustar header y longitud
    if header is None:
        header = [f"Col_{i+1}" for i in range(len(data_rows[0]))]
    else:
        if len(header) < len(data_rows[0]):
            header += [f"Col_{i+1}" for i in range(len(header), len(data_rows[0]))]
        else:
            header = header[:len(data_rows[0])]
    
    df = pd.DataFrame(data_rows, columns=header)
    
    # 6. FORZAR NOMBRES DE COLUMNAS ÚNICOS (evita el InvalidIndexError)
    if not df.columns.is_unique:
        cols = pd.Series(df.columns)
        for dup in cols[cols.duplicated()].unique():
            indices = cols[cols == dup].index
            for i, idx in enumerate(indices):
                if i == 0:
                    continue
                df.columns.values[idx] = f"{dup}_{i}"
    
    # 7. Convertir columnas numéricas
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass
    
    return df

def asegurar_columnas_unicas(df):
    """Convierte los nombres de columnas de un DataFrame en únicos."""
    cols = pd.Series(df.columns).astype(str)
    if cols.is_unique:
        return df
    
    nuevas_columnas = []
    contador = {}
    for c in cols:
        if c in contador:
            contador[c] += 1
            nuevas_columnas.append(f"{c}_{contador[c]}")
        else:
            contador[c] = 1
            nuevas_columnas.append(c)
    
    df.columns = nuevas_columnas
    return df

In [ ]:
# Celda 9 - Obtener datos de UEFA

def ejecutar_scraping():
    """
    Función que orquesta la obtención de datos de UEFA y AFC.
    Guarda los CSVs directamente en la ruta definida en BASE_DATA_DIR.
    """
    # Aseguramos que la variable global esté disponible
    global BASE_DATA_DIR  

    urls = {
        'uefa': 'https://kassiesa.net/uefa/data/method5/trank2025.html',
        'afc': 'https://footyrankings.github.io/cpts2025/'
    }
    
    dataframes = {}
    
    for name, url in urls.items():
        print(f"\n--- Procesando {name.upper()} ---")
        html = fetch_html(url)
        if html is None:
            print(f"  [ERROR] No se pudo obtener HTML de {url}")
            continue
        
        try:
            if name == 'uefa':
                df = parse_uefa(html)
            else:  # afc
                df = parse_afc(html)
            
            dataframes[name] = df
            print(f"  → {len(df)} registros extraídos correctamente.")
            
            # --- USO DE SU VARIABLE GLOBAL ---
            # Construimos la ruta completa: BASE_DATA_DIR + nombre_archivo
            archivo_salida = f"{BASE_DATA_DIR}/{name}_ranking_2025.csv"
            df.to_csv(archivo_salida, index=False, encoding='utf-8-sig')
            print(f"  → Archivo guardado en: {archivo_salida}")
            
        except Exception as e:
            print(f"  [ERROR] Falló el parseo de {name}: {e}")
    
    # Guardar el combinado (opcional, pero útil para su otro notebook)
     # Guardar el combinado (CON MANEJO ROBUSTO DE COLUMNAS)
    if dataframes:
        dfs_limpios = []
        for name, df in dataframes.items():
            # 1. Hacemos una copia para no alterar los originales
            df_copia = df.copy()
            # 2. Saneamos las columnas (eliminamos duplicados)
            df_copia = asegurar_columnas_unicas(df_copia)
            # 3. Añadimos la columna de origen
            df_copia['fuente'] = name
            dfs_limpios.append(df_copia)
        
        # 4. Concatenamos de forma segura
        combined = pd.concat(dfs_limpios, ignore_index=True)
        archivo_combinado = f"{BASE_DATA_DIR}/combined_rankings_2025.csv"
        combined.to_csv(archivo_combinado, index=False, encoding='utf-8-sig')
        print(f"\n→ Combinado guardado en: {archivo_combinado}")
    
    return dataframes

# --- EJECUTAR ---
datos_obtenidos = ejecutar_scraping()


--- Procesando UEFA ---
  → 482 registros extraídos correctamente.
  → Archivo guardado en: /content/drive/MyDrive/TEG02-2025Arellano/Data/uefa_ranking_2025.csv

--- Procesando AFC ---


/tmp/ipykernel_2493/2661898959.py:148: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


  → 6 registros extraídos correctamente.
  → Archivo guardado en: /content/drive/MyDrive/TEG02-2025Arellano/Data/afc_ranking_2025.csv

→ Combinado guardado en: /content/drive/MyDrive/TEG02-2025Arellano/Data/combined_rankings_2025.csv


In [9]:
# Diagnóstico de los archivos guardados
print("="*50)
print("DIAGNÓSTICO UEFA")
uefa_df = pd.read_csv(f"{BASE_DATA_DIR}/uefa_ranking_2025.csv")
print(f"Filas: {len(uefa_df)}")
print(f"Columnas: {uefa_df.columns.tolist()}")
print(uefa_df.head(3))

print("\n"+"="*50)
print("DIAGNÓSTICO AFC")
afc_df = pd.read_csv(f"{BASE_DATA_DIR}/afc_ranking_2025.csv")
print(f"Filas: {len(afc_df)}")
print(f"Columnas: {afc_df.columns.tolist()}")
print(afc_df.head(10))  # Ver las primeras 10 para ver si hay más de 5

DIAGNÓSTICO UEFA
Filas: 482
Columnas: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', '20/21', '21/22', '22/23', '23/24', '24/25', 'TotalPoints', 'CountryPart']
   Unnamed: 0  Unnamed: 1       Unnamed: 2 Unnamed: 3  20/21  21/22  22/23  \
0         1.0         NaN      Real Madrid        Esp   26.0   30.0   29.0   
1         2.0         NaN  Manchester City        Eng   35.0   27.0   33.0   
2         3.0         NaN  Bayern MÃ¼nchen        Ger   27.0   26.0   27.0   

   23/24  24/25  TotalPoints  CountryPart  
0   34.0  24.50       143.50       18.890  
1   28.0  14.75       137.75       23.039  
2   28.0  27.25       135.25       17.266  

DIAGNÓSTICO AFC
Filas: 6
Columnas: ['Group and knockout stage', 'Group and knockout stage_1', '16.400', 'Col_4', 'Col_5', 'Col_6', 'Col_7', 'Col_8', 'Col_9']
   Group and knockout stage    Group and knockout stage_1 16.400 Col_4 Col_5  \
0                         3                        UAEUAE    Cup     W     D   
1                     

In [ ]:
# Agregamos nuevas librerias, mover arriba
import os
import re
import pandas as pd
import numpy as np
import chardet
from google.colab import drive
from pathlib import Path

# Instalación de librerías específicas (ejecutar en Colab)
!pip install pdfplumber tabula-py pypdf2 --quiet

import pdfplumber
import tabula
import PyPDF2  # Fallback para texto plano

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 65.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 113.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 105.0 MB/s eta 0:00:0000:01


In [ ]:
import os
import pandas as pd
import numpy as np
import chardet
from pathlib import Path

# Ruta específica
#BASE_DATA_DIR = "/content/drive/MyDrive/TEG02-2025Arellano/Data"
file_path = os.path.join(BASE_DATA_DIR, "uefa_ranking_2025.csv")

# --- 1. Diagnóstico de encoding y separador ---
with open(file_path, 'rb') as f:
    raw_sample = f.read(10000)
    encoding_result = chardet.detect(raw_sample)
    detected_encoding = encoding_result['encoding']
    print(f"[Diagnóstico] Encoding inferido: {detected_encoding} (confianza: {encoding_result['confidence']:.2f})")

# Leemos las primeras líneas en modo texto para ver el separador
with open(file_path, 'r', encoding=detected_encoding) as f:
    first_lines = [f.readline() for _ in range(5)]
    print("\n[Vista previa de primeras líneas]:")
    for i, line in enumerate(first_lines):
        print(f"Línea {i+1}: {line[:100]}...")  # trunca para no saturar

# Inferimos separador: si encuentra tabulador -> '\t', si encuentra punto y coma -> ';', si no -> ','
sample_line = first_lines[0] if first_lines else ""
if '\t' in sample_line:
    sep = '\t'
elif ';' in sample_line:
    sep = ';'
else:
    sep = ','  # default
print(f"[Diagnóstico] Separador inferido: '{sep}'")

# --- 2. Carga con parámetros robustos ---
try:
    df_raw = pd.read_csv(
        file_path,
        encoding=detected_encoding,
        sep=sep,
        decimal=',' if sep == ';' else '.',  # si usa ';' suele usar coma como decimal
        engine='python',  # más flexible para separadores complejos
        skip_blank_lines=True,
        thousands=None  # lo dejamos None para detectar manualmente
    )
    print(f"\n[OK] Carga exitosa. Shape: {df_raw.shape}")
except Exception as e:
    print(f"[ERROR] Falló la carga estándar: {e}. Intentando con encoding 'latin-1' y sep=','...")
    df_raw = pd.read_csv(file_path, encoding='latin-1', sep=',', engine='python')


    # --- 3. Estandarización de nombres de columnas ---
def standardize_colname(col):
    # Convertir a minúsculas, quitar acentos (mapeo básico), reemplazar espacios y puntuación por _
    import unicodedata
    col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')
    col = col.lower().strip()
    col = re.sub(r'[^a-z0-9]', '_', col)  # solo letras, números y _
    col = re.sub(r'_+', '_', col)         # múltiples _ a uno solo
    return col

# Aplicar a todas las columnas
df_raw.columns = [standardize_colname(c) for c in df_raw.columns]

# Mostrar los nombres estandarizados
print("\n[Nombres de columnas estandarizados]:")
print(df_raw.columns.tolist())

# --- 4. Limpieza de filas basura (p.ej., filas de "Average" o "Total" al final) ---
# Heurística: si alguna columna clave (ej: 'rank' o 'position') tiene strings no numéricos
# y la fila parece un agregado, la eliminamos.
# Para este caso, asumimos que la columna de posición debe ser entera.
# Buscamos la columna que contenga 'rank' o 'position' o 'pos'
rank_col = None
for col in df_raw.columns:
    if 'rank' in col or 'position' in col or 'pos' in col:
        rank_col = col
        break

if rank_col:
    # Intentamos convertir a numérico, los que no se puedan serán considerados basura
    df_raw[rank_col + '_num'] = pd.to_numeric(df_raw[rank_col], errors='coerce')
    # Eliminamos filas donde la conversión falló (son headers o totales)
    df_raw = df_raw.dropna(subset=[rank_col + '_num'])
    # Eliminamos la columna auxiliar
    df_raw.drop(columns=[rank_col + '_num'], inplace=True)
    print(f"[Limpieza] Filas eliminadas por no tener valor numérico en '{rank_col}': {len(df_raw) - len(df_raw)}") # ajustar lógica

# --- 5. Renombramiento definitivo a un estándar del proyecto ---
# Definimos un mapping semántico (ajústalo según lo que veas en df_raw.columns)
mapping = {}
for col in df_raw.columns:
    if 'rank' in col and 'pos' not in col:
        mapping[col] = 'posicion_uefa'
    elif 'country' in col or 'nation' in col or 'team' in col or 'club' in col:
        mapping[col] = 'pais_equipo'
    elif 'point' in col or 'pts' in col or 'score' in col:
        mapping[col] = 'puntos_uefa'
    elif 'coeff' in col or 'coef' in col:
        mapping[col] = 'coeficiente'
    elif 'year' in col or 'season' in col:
        mapping[col] = 'temporada'

# Aplicamos solo las columnas que existen
df_estandar = df_raw.rename(columns={k: v for k, v in mapping.items() if k in df_raw.columns})

# Si alguna columna crítica no se renombró, la dejamos con su nombre estandarizado
print("\n[Columnas finales del DataFrame estandarizado]:")
print(df_estandar.columns.tolist())

[Diagnóstico] Encoding inferido: UTF-8-SIG (confianza: 1.00)

[Vista previa de primeras líneas]:
Línea 1: ,,,,20/21,21/22,22/23,23/24,24/25,TotalPoints,CountryPart
...
Línea 2: 1,,Real Madrid,Esp,26.00,30.00,29.00,34.00,24.50,143.500,18.890
...
Línea 3: 2,,Manchester City,Eng,35.00,27.00,33.00,28.00,14.75,137.750,23.039
...
Línea 4: 3,,Bayern MÃ¼nchen,Ger,27.00,26.00,27.00,28.00,27.25,135.250,17.266
...
Línea 5: 4,,Liverpool,Eng,24.00,33.00,19.00,20.00,29.50,125.500,23.039
...
[Diagnóstico] Separador inferido: ','

[OK] Carga exitosa. Shape: (482, 11)

[Nombres de columnas estandarizados]:
['unnamed_0', 'unnamed_1', 'unnamed_2', 'unnamed_3', '20_21', '21_22', '22_23', '23_24', '24_25', 'totalpoints', 'countrypart']

[Columnas finales del DataFrame estandarizado]:
['unnamed_0', 'unnamed_1', 'unnamed_2', 'unnamed_3', '20_21', '21_22', '22_23', '23_24', '24_25', 'puntos_uefa', 'pais_equipo']


In [ ]:
# ==========================================
# BLOQUE 4: ESTRUCTURA FINAL Y LIMPIEZA DE UEFA
# ==========================================

# 1. Renombrar según tu especificación explícita
rename_map = {
    'unnamed_0': 'Rank',
    'unnamed_2': 'Team',
    'unnamed_3': 'Country'
}
df_estandar.rename(columns=rename_map, inplace=True)

# 2. Eliminar la columna basura (unnamed_1) - la 2da columna
if 'unnamed_1' in df_estandar.columns:
    df_estandar.drop(columns=['unnamed_1'], inplace=True)
    print("[Limpieza] Columna 'unnamed_1' (vacía) eliminada correctamente.")

# 3. Verificar que el orden de columnas es el esperado
print("\n[✅ Columnas finales del DataFrame]:")
print(df_estandar.columns.tolist())
# Debe mostrar: ['Rank', 'Team', 'Country', '20_21', '21_22', '22_23', '23_24', '24_25', 'puntos_uefa', 'pais_equipo']

# 4. Conversión forzada de tipos (fundamental para estadística)
#    - Rank debe ser entero
#    - Temporadas y puntos deben ser float (los puntos tienen decimales)
numeric_cols = ['Rank', '20_21', '21_22', '22_23', '23_24', '24_25', 'puntos_uefa', 'pais_equipo']
for col in numeric_cols:
    if col in df_estandar.columns:
        df_estandar[col] = pd.to_numeric(df_estandar[col], errors='coerce')

# Convertimos Rank a entero (nullable, por si hay algún NA)
df_estandar['Rank'] = df_estandar['Rank'].astype('Int64')

# 5. Validación semántica: ¿Coinciden los totales con la suma de temporadas?
#    Esto es un control de integridad matemática para tu apéndice.
year_cols = ['20_21', '21_22', '22_23', '23_24', '24_25']
# Calculamos la suma de las 5 temporadas y la comparamos con 'puntos_uefa'
if all(col in df_estandar.columns for col in year_cols):
    df_estandar['suma_temporadas'] = df_estandar[year_cols].sum(axis=1)
    # Redondeamos a 3 decimales para evitar errores de floating point
    df_estandar['check_total'] = np.isclose(df_estandar['suma_temporadas'], 
                                            df_estandar['puntos_uefa'], 
                                            rtol=1e-3)
    print(f"\n[Integridad de datos] ¿Coinciden las sumas de temporadas con TotalPoints?")
    print(f"  - Coinciden: {df_estandar['check_total'].sum()} filas")
    print(f"  - Discrepan: {(~df_estandar['check_total']).sum()} filas (revisar si hay redondeos)")
    # Eliminamos columnas auxiliares para no saturar el DF final
    df_estandar.drop(columns=['suma_temporadas', 'check_total'], inplace=True)

# 6. Vista previa definitiva para que valides
print("\n[✅ Vista previa de los primeros 5 registros Estandarizados]:")
display(df_estandar.head())

print("\n[📊 Resumen de tipos de datos]:")
print(df_estandar.dtypes)

print(f"\n[📏 Dimensión final del DataFrame]: {df_estandar.shape[0]} filas y {df_estandar.shape[1]} columnas.")

[Limpieza] Columna 'unnamed_1' (vacía) eliminada correctamente.

[✅ Columnas finales del DataFrame]:
['Rank', 'Team', 'Country', '20_21', '21_22', '22_23', '23_24', '24_25', 'puntos_uefa', 'pais_equipo']

[Integridad de datos] ¿Coinciden las sumas de temporadas con TotalPoints?
  - Coinciden: 427 filas
  - Discrepan: 55 filas (revisar si hay redondeos)

[✅ Vista previa de los primeros 5 registros Estandarizados]:


,Rank,Team,Country,20_21,21_22,22_23,23_24,24_25,puntos_uefa,pais_equipo
0,1,Real Madrid,Esp,26.0,30.0,29.0,34.0,24.50,143.50,18.890
1,2,Manchester City,Eng,35.0,27.0,33.0,28.0,14.75,137.75,23.039
2,3,Bayern MÃ¼nchen,Ger,27.0,26.0,27.0,28.0,27.25,135.25,17.266
3,4,Liverpool,Eng,24.0,33.0,19.0,20.0,29.50,125.50,23.039
4,5,Paris Saint-Germain,Fra,24.0,19.0,19.0,23.0,33.50,118.50,14.618



[📊 Resumen de tipos de datos]:
Rank             Int64
Team            object
Country         object
20_21          float64
21_22          float64
22_23          float64
23_24          float64
24_25          float64
puntos_uefa    float64
pais_equipo    float64
dtype: object

[📏 Dimensión final del DataFrame]: 482 filas y 10 columnas.


In [13]:
# ==========================================
# BLOQUE 5 (CORREGIDO): FILTRADO POR PAÍSES
# ==========================================

# 1. Limpieza EXTREMA de la columna Country
# Convertimos TODO a string, eliminamos espacios y reemplazamos 'nan' (string) por ''
df_estandar['Country'] = df_estandar['Country'].astype(str).str.strip()
df_estandar['Country'] = df_estandar['Country'].replace('nan', '')

# 2. Eliminamos filas donde Country quedó vacío (son basura 100% segura)
df_estandar = df_estandar[df_estandar['Country'] != ''].copy()
print(f"[Limpieza] Filas con Country vacío eliminadas. Shape actual: {df_estandar.shape}")

# 3. Diagnóstico de países (AHORA SÍ FUNCIONA)
print("\n[Diagnóstico de país] Valores únicos en la columna 'Country' (primeros 30):")
unique_countries = df_estandar['Country'].unique()
# Ya todos son strings, sorted funciona
print(sorted(unique_countries)[:30])

# 4. Definir los códigos objetivo (¡OJO! puede ser 'FRA' en lugar de 'Fra')
target_countries = ['Esp', 'Eng', 'Ger', 'Fra', 'Ita', 'Gre']
# También agregamos las variantes en mayúsculas por si acaso
target_countries_extended = target_countries + [c.upper() for c in target_countries]

print("\n[Conteo pre-filtro] Cantidad de equipos por país objetivo (incluyendo variantes):")
for c in target_countries_extended:
    count = (df_estandar['Country'] == c).sum()
    if count > 0:
        print(f"  - '{c}': {count} equipos")

# 5. Aplicar el filtro (usamos la lista extendida para no perder ningún equipo)
df_filtrado = df_estandar[df_estandar['Country'].isin(target_countries_extended)].copy()

# 6. Resetear el índice (fundamental para modelos estadísticos)
df_filtrado.reset_index(drop=True, inplace=True)

print(f"\n[✅ Filtro aplicado] Shape original: {df_estandar.shape[0]} filas → Shape nuevo: {df_filtrado.shape[0]} filas")

[Limpieza] Filas con Country vacío eliminadas. Shape actual: (427, 10)

[Diagnóstico de país] Valores únicos en la columna 'Country' (primeros 30):
['Alb', 'And', 'Arm', 'Aut', 'Azb', 'Bel', 'Bls', 'Bos', 'Bul', 'Cro', 'Cyp', 'Cze', 'Den', 'Eng', 'Esp', 'Est', 'Far', 'Fin', 'Fra', 'Geo', 'Ger', 'Gib', 'Gre', 'Hun', 'Irl', 'Isl', 'Isr', 'Ita', 'Kaz', 'Kos']

[Conteo pre-filtro] Cantidad de equipos por país objetivo (incluyendo variantes):
  - 'Esp': 11 equipos
  - 'Eng': 11 equipos
  - 'Ger': 13 equipos
  - 'Fra': 12 equipos
  - 'Ita': 9 equipos
  - 'Gre': 6 equipos

[✅ Filtro aplicado] Shape original: 427 filas → Shape nuevo: 62 filas


In [14]:
# ==========================================
# BLOQUE 6: ANÁLISIS DEL RANK Y DECISIÓN
# ==========================================

# Mostramos un sample para que veas los saltos en el Rank
print("\n[Vista previa del Rank después del filtro (primeros 15 equipos)]:")
display(df_filtrado[['Rank', 'Team', 'Country']].head(15))

print("\n[Distribución de países en la muestra filtrada]:")
print(df_filtrado['Country'].value_counts())

# Verificar si hay duplicados de Team (por si algún equipo cambió de país)
print(f"\n[Duplicados en Team]: {df_filtrado['Team'].duplicated().sum()} equipos repetidos (revisar si son homónimos)")

# --- Decisión sobre el Rank ---
# Voy a proponer mantener el Rank original, pero creo una columna 'Rank_local' por si la necesitas.
df_filtrado['Rank_local'] = df_filtrado['Rank'].rank(method='dense').astype('Int64')

print("\n[Comparativa de Ranks]")
print(f"Rango del Rank original: {df_filtrado['Rank'].min()} a {df_filtrado['Rank'].max()}")
print(f"Rango del Rank_local (denso): {df_filtrado['Rank_local'].min()} a {df_filtrado['Rank_local'].max()}")

# Exportamos una vista limpia para que valides
display(df_filtrado[['Rank', 'Rank_local', 'Team', 'Country', 'puntos_uefa']].head(10))


[Vista previa del Rank después del filtro (primeros 15 equipos)]:


,Rank,Team,Country
0,1,Real Madrid,Esp
1,2,Manchester City,Eng
2,3,Bayern MÃ¼nchen,Ger
3,4,Liverpool,Eng
4,5,Paris Saint-Germain,Fra
5,6,Internazionale,Ita
6,7,Chelsea,Eng
7,8,Borussia Dortmund,Ger
8,9,AS Roma,Ita
9,10,FC Barcelona,Esp



[Distribución de países en la muestra filtrada]:
Country
Ger    13
Fra    12
Eng    11
Esp    11
Ita     9
Gre     6
Name: count, dtype: int64

[Duplicados en Team]: 0 equipos repetidos (revisar si son homónimos)

[Comparativa de Ranks]
Rango del Rank original: 1 a 195
Rango del Rank_local (denso): 1 a 44


,Rank,Rank_local,Team,Country,puntos_uefa
0,1,1,Real Madrid,Esp,143.50
1,2,2,Manchester City,Eng,137.75
2,3,3,Bayern MÃ¼nchen,Ger,135.25
3,4,4,Liverpool,Eng,125.50
4,5,5,Paris Saint-Germain,Fra,118.50
5,6,6,Internazionale,Ita,116.25
6,7,7,Chelsea,Eng,109.00
7,8,8,Borussia Dortmund,Ger,106.75
8,9,9,AS Roma,Ita,104.50
9,10,10,FC Barcelona,Esp,103.25


In [15]:
import os
import pandas as pd
import numpy as np
import chardet
import re

# Ruta del archivo AFC
BASE_DATA_DIR = "/content/drive/MyDrive/TEG02-2025Arellano/Data"
file_path_afc = os.path.join(BASE_DATA_DIR, "RankingAFC.csv")

# --- 1. Diagnóstico de encoding ---
with open(file_path_afc, 'rb') as f:
    raw_sample = f.read(10000)
    encoding_result = chardet.detect(raw_sample)
    detected_encoding = encoding_result['encoding']
    print(f"[Diagnóstico AFC] Encoding inferido: {detected_encoding} (confianza: {encoding_result['confidence']:.2f})")

# --- 2. Vista previa de primeras líneas (para ver separador y estructura) ---
with open(file_path_afc, 'r', encoding=detected_encoding) as f:
    first_lines = [f.readline() for _ in range(6)]
    print("\n[Vista previa de primeras líneas (AFC)]:")
    for i, line in enumerate(first_lines):
        print(f"Línea {i+1}: {line[:120].rstrip()}...")  # trunca

# --- 3. Inferir separador ---
sample_line = first_lines[0] if first_lines else ""
if '\t' in sample_line:
    sep = '\t'
elif ';' in sample_line:
    sep = ';'
else:
    sep = ','
print(f"[Diagnóstico AFC] Separador inferido: '{sep}'")
df_afc=pd.read_csv(
        file_path_afc,
        encoding=detected_encoding,
        sep=sep,
        decimal=',' if sep == ';' else '.',  # si usa ';' suele usar coma como decimal
        engine='python',  # más flexible para separadores complejos
        skip_blank_lines=True,
        thousands=None  # lo dejamos None para detectar manualmente
)
print(f"Dataframe AFC listo")

[Diagnóstico AFC] Encoding inferido: UTF-8-SIG (confianza: 1.00)

[Vista previa de primeras líneas (AFC)]:
Línea 1: Country,Team,Cup,W,D,L,Bonus,Points...
Línea 2: KSA,AlAhli,CLE,12,1,0,10.55,47.5...
Línea 3: KSA,AlHilal,CLE,9,1,2,9,37...
Línea 4: KSA,AlNassr,CLE,7,3,2,9,33...
Línea 5: KSA,AlTaawoun,CL2,6,4,2,9,20.667...
Línea 6: JPN,KawasakiF,CLE,7,1,4,10.5,34.597...
[Diagnóstico AFC] Separador inferido: ','
Dataframe AFC listo


In [16]:
df_uefa=df_filtrado
print(df_uefa.columns.tolist())
print(df_afc.columns.tolist())

['Rank', 'Team', 'Country', '20_21', '21_22', '22_23', '23_24', '24_25', 'puntos_uefa', 'pais_equipo', 'Rank_local']
['Country', 'Team', 'Cup', 'W', 'D', 'L', 'Bonus', 'Points']


In [ ]:
# ==========================================
# CELDA 4: DROP DE COLUMNAS (MANUAL)
# ==========================================

# --- Define las columnas a eliminar para UEFA ---
# (Ejemplo: si quieres borrar la columna 'unnamed_1' y 'countrypart' que ya no necesitas)
cols_drop_uefa = ['20_21','21_22','22_23','23_24','24_25']  # <--- CAMBIA AQUÍ según lo que veas

# --- Define las columnas a eliminar para AFC ---
# (Ejemplo: quitar 'cup', 'w', 'd', 'l', 'bonus' si solo quieres puntos y equipo/país)
cols_drop_afc = ['Cup', 'W', 'D', 'L', 'Bonus',]  # <--- CAMBIA AQUÍ

# --- Aplicar drops (solo si existen) ---
if cols_drop_uefa:
    cols_to_drop_uefa = [col for col in cols_drop_uefa if col in df_uefa.columns]
    if cols_to_drop_uefa:
        df_uefa.drop(columns=cols_to_drop_uefa, inplace=True)
        print(f"[UEFA] Columnas eliminadas: {cols_to_drop_uefa}")
    else:
        print("[UEFA] Ninguna de las columnas especificadas existe. No se eliminó nada.")

if cols_drop_afc:
    cols_to_drop_afc = [col for col in cols_drop_afc if col in df_afc.columns]
    if cols_to_drop_afc:
        df_afc.drop(columns=cols_to_drop_afc, inplace=True)
        print(f"[AFC] Columnas eliminadas: {cols_to_drop_afc}")
    else:
        print("[AFC] Ninguna de las columnas especificadas existe. No se eliminó nada.")

# --- Mostrar las columnas finales ---
print("\n[Columnas finales - UEFA]:", df_uefa.columns.tolist())
print("[Columnas finales - AFC]:", df_afc.columns.tolist())

[UEFA] Columnas eliminadas: ['20_21', '21_22', '22_23', '23_24', '24_25']
[AFC] Columnas eliminadas: ['Cup', 'W', 'D', 'L', 'Bonus']

[Columnas finales - UEFA]: ['Rank', 'Team', 'Country', 'puntos_uefa', 'pais_equipo', 'Rank_local']
[Columnas finales - AFC]: ['Country', 'Team', 'Points']


In [ ]:
# ==========================================
# CELDA 1: FUNCIÓN DE CARGA PARA ELO (TXT)
# ==========================================
import os
import pandas as pd
import numpy as np
import chardet
import re

def load_elo_txt(file_path):
    """
    Carga archivos ELO en formato TXT.
    Detecta automáticamente separadores (tabulador, espacios múltiples, coma).
    Elimina filas basura (metadatos, pies de página, líneas vacías).
    """
    # 1. Detectar encoding
    with open(file_path, 'rb') as f:
        raw_sample = f.read(10000)
        encoding_result = chardet.detect(raw_sample)
        enc = encoding_result['encoding']
        print(f"[Diagnóstico ELO] Encoding: {enc} (confianza: {encoding_result['confidence']:.2f})")
    
    # 2. Leer primeras líneas para inspección
    with open(file_path, 'r', encoding=enc) as f:
        lines = f.readlines()
    
    print("\n[Vista previa de primeras líneas (ELO)]:")
    for i in range(min(10, len(lines))):
        print(f"Línea {i+1}: {lines[i].rstrip()}")
    
    # 3. Intentar cargar con pandas (auto-detecta separador)
    # Usamos sep=None para que pandas infiera el separador automáticamente
    try:
        # Leemos con engine='python' para mayor flexibilidad
        df = pd.read_csv(
            file_path,
            encoding=enc,
            sep=None,          # Auto-detección de separador
            engine='python',
            skip_blank_lines=True,
            on_bad_lines='warn'
        )
        print(f"[OK] Carga con auto-detección exitosa. Shape: {df.shape}")
    except Exception as e:
        print(f"[WARN] Auto-detección falló ({e}). Intentando con separador por espacios múltiples...")
        try:
            df = pd.read_csv(
                file_path,
                encoding=enc,
                sep=r'\s+',      # Uno o más espacios/tabuladores
                engine='python',
                skip_blank_lines=True,
                on_bad_lines='warn'
            )
            print(f"[OK] Carga con separador '\\s+' exitosa. Shape: {df.shape}")
        except Exception as e2:
            print(f"[ERROR] Falló también con '\\s+': {e2}. Intentando con coma (','), fallback total.")
            df = pd.read_csv(file_path, encoding='latin-1', sep=',', engine='python')
            print(f"[OK] Carga fallback con coma. Shape: {df.shape}")
    
    # 4. Limpieza de filas con demasiados nulos (basura)
    df.dropna(thresh=int(df.shape[1] * 0.5), inplace=True)
    
    # 5. Estandarizar nombres de columnas (snake_case)
    def standardize(col):
        col = str(col).lower().strip()
        col = re.sub(r'[^a-z0-9]', '_', col)
        col = re.sub(r'_+', '_', col)
        return col
    
    df.columns = [standardize(c) for c in df.columns]
    
    # 6. Intentar identificar la columna de Rank y eliminar filas donde Rank no sea numérico
    rank_col_candidates = [col for col in df.columns if 'rank' in col or 'pos' in col]
    if rank_col_candidates:
        rank_col = rank_col_candidates[0]
        # Convertimos a numérico, los que no se puedan se vuelven NaN
        df[rank_col] = pd.to_numeric(df[rank_col], errors='coerce')
        df = df.dropna(subset=[rank_col])
        # Convertimos Rank a entero si es posible
        df[rank_col] = df[rank_col].astype('Int64')
        print(f"[Limpieza] Filas sin Rank numérico eliminadas. Shape actual: {df.shape}")
    
    print(f"\n[Nombres de columnas estandarizados]:\n{df.columns.tolist()}")
    print("\n[Primeras 5 filas (vista previa)]:")
    display(df.head())
    
    return df

In [19]:
# ==========================================
# CELDA 2: CARGA DE ELO20260106.txt
# ==========================================
BASE_DATA_DIR = "/content/drive/MyDrive/TEG02-2025Arellano/Data"
file_elo1 = os.path.join(BASE_DATA_DIR, "ELO20260106.txt")

df_elo_jan = load_elo_txt(file_elo1)
# Añadimos una columna con la fecha del snapshot (extraída del nombre)
df_elo_jan['fecha_snapshot'] = '2026-01-06'
print("\n[ELO 2026-01-06] DataFrame listo.")

[Diagnóstico ELO] Encoding: utf-8 (confianza: 0.99)

[Vista previa de primeras líneas (ELO)]:
Línea 1: Rank,Club,Elo,Last Match,Coach
Línea 2: 1,ENG 1 Arsenal,2054,2026-01-03,"Mikel Arteta (since 2019-12-22)"
Línea 3: 2,GER 2 Bayern,1993,2025-12-21,"Vincent Kompany (since 2024-07-01)"
Línea 4: 3,ENG 3 Man City,1993,2026-01-04,"Pep Guardiola (since 2016-07-01)"
Línea 5: 4,FRA 4 Paris SG,1968,2026-01-04,"Luis Enrique (since 2023-07-05)"
Línea 6: 5,ESP 5 Barcelona,1945,2026-01-03,"Hansi Flick (since 2024-07-01)"
Línea 7: 6,ENG 6 Liverpool,1938,2026-01-04,"Arne Slot (since 2024-06-01)"
Línea 8: 7,ITA 7 Inter,1923,2026-01-04,"Cristian Chivu (since 2025-06-09)"
Línea 9: 8,ENG 8 Aston Villa,1921,2026-01-03,"Unai Emery (since 2022-11-01)"
Línea 10: 9,ESP 9 Real Madrid,1910,2026-01-04,"Xabi Alonso (since 2025-06-01)"
[OK] Carga con auto-detección exitosa. Shape: (630, 5)
[Limpieza] Filas sin Rank numérico eliminadas. Shape actual: (630, 5)

[Nombres de columnas estandarizados]:
['rank', 'club',

,rank,club,elo,last_match,coach
0,1,ENG 1 Arsenal,2054,2026-01-03,Mikel Arteta (since 2019-12-22)
1,2,GER 2 Bayern,1993,2025-12-21,Vincent Kompany (since 2024-07-01)
2,3,ENG 3 Man City,1993,2026-01-04,Pep Guardiola (since 2016-07-01)
3,4,FRA 4 Paris SG,1968,2026-01-04,Luis Enrique (since 2023-07-05)
4,5,ESP 5 Barcelona,1945,2026-01-03,Hansi Flick (since 2024-07-01)



[ELO 2026-01-06] DataFrame listo.


In [20]:
# ==========================================
# CELDA 3: CARGA DE ELO20250802.txt
# ==========================================
file_elo2 = os.path.join(BASE_DATA_DIR, "ELO20250802.txt")

df_elo_aug = load_elo_txt(file_elo2)
df_elo_aug['fecha_snapshot'] = '2025-08-02'
print("\n[ELO 2025-08-02] DataFrame listo.")

[Diagnóstico ELO] Encoding: utf-8 (confianza: 0.99)

[Vista previa de primeras líneas (ELO)]:
Línea 1: Rank,Country,Club,Elo,Last Match,Coach
Línea 2: 1,ENG 1 Liverpool,1993,2025-05-25,Arne Slot (since 2024-06-01)
Línea 3: 2,ENG 2 Arsenal,1993,2025-05-25,Mikel Arteta (since 2019-12-22)
Línea 4: 3,FRA 3 Paris SG,1975,2025-05-31,Luis Enrique (since 2023-07-05)
Línea 5: 4,ENG 4 Man City,1960,2025-05-25,Pep Guardiola (since 2016-07-01)
Línea 6: 5,ESP 5 Barcelona,1945,2025-05-25,Hansi Flick (since 2024-07-01)
Línea 7: 6,ESP 6 Real Madrid,1936,2025-05-24,Xabi Alonso (since 2025-06-01)
Línea 8: 7,ITA 7 Inter,1933,2025-05-31,Cristian Chivu (since 2025-06-09)
Línea 9: 8,GER 8 Bayern,1919,2025-05-17,Vincent Kompany (since 2024-07-01)
Línea 10: 9,ENG 9 Chelsea,1903,2025-05-28,Enzo Maresca (since 2024-07-01)
[OK] Carga con auto-detección exitosa. Shape: (630, 6)
[Limpieza] Filas sin Rank numérico eliminadas. Shape actual: (630, 6)

[Nombres de columnas estandarizados]:
['rank', 'country', 'club', 

,rank,country,club,elo,last_match,coach
0,1,ENG 1 Liverpool,1993,2025-05-25,Arne Slot (since 2024-06-01),NaN
1,2,ENG 2 Arsenal,1993,2025-05-25,Mikel Arteta (since 2019-12-22),NaN
2,3,FRA 3 Paris SG,1975,2025-05-31,Luis Enrique (since 2023-07-05),NaN
3,4,ENG 4 Man City,1960,2025-05-25,Pep Guardiola (since 2016-07-01),NaN
4,5,ESP 5 Barcelona,1945,2025-05-25,Hansi Flick (since 2024-07-01),NaN



[ELO 2025-08-02] DataFrame listo.


In [ ]:
# ==========================================
# CELDA 4: DROP DE COLUMNAS (MANUAL)
# ==========================================

# --- Columnas a eliminar en ELO 2026-01-06 ---
# Ejemplo: si no quieres 'change' porque no la vas a usar en el modelo de clasificación:
cols_drop_elo_jan = ['last_match', 'coach','fecha_snapshot']   # <--- CAMBIA AQUÍ según tu análisis

# --- Columnas a eliminar en ELO 2025-08-02 ---
cols_drop_elo_aug = ['last_match', 'coach','fecha_snapshot',]   # <--- CAMBIA AQUÍ

# --- Aplicar drops (solo si existen) ---
if cols_drop_elo_jan:
    cols_to_drop = [col for col in cols_drop_elo_jan if col in df_elo_jan.columns]
    if cols_to_drop:
        df_elo_jan.drop(columns=cols_to_drop, inplace=True)
        print(f"[ELO 2026-01-06] Columnas eliminadas: {cols_to_drop}")
    else:
        print("[ELO 2026-01-06] Ninguna de las columnas especificadas existe.")

if cols_drop_elo_aug:
    cols_to_drop = [col for col in cols_drop_elo_aug if col in df_elo_aug.columns]
    if cols_to_drop:
        df_elo_aug.drop(columns=cols_to_drop, inplace=True)
        print(f"[ELO 2025-08-02] Columnas eliminadas: {cols_to_drop}")
    else:
        print("[ELO 2025-08-02] Ninguna de las columnas especificadas existe.")

# --- Mostrar columnas finales ---
print("\n[Columnas finales - ELO 2026-01-06]:", df_elo_jan.columns.tolist())
print("[Columnas finales - ELO 2025-08-02]:", df_elo_aug.columns.tolist())

[ELO 2026-01-06] Columnas eliminadas: ['last_match', 'coach', 'fecha_snapshot']
[ELO 2025-08-02] Columnas eliminadas: ['last_match', 'coach', 'fecha_snapshot']

[Columnas finales - ELO 2026-01-06]: ['rank', 'club', 'elo']
[Columnas finales - ELO 2025-08-02]: ['rank', 'country', 'club', 'elo']


In [22]:
# 1. Intentamos convertir 'club' a numérico para confirmar
try:
    df_elo_aug['club_num'] = pd.to_numeric(df_elo_aug['club'], errors='coerce')
    print(f"\n[Verificación] 'club' es numérico en {df_elo_aug['club_num'].notna().sum()} de {len(df_elo_aug)} filas.")
    # Si la mayoría son numéricos, confirmamos que 'club' es realmente ELO.
except Exception as e:
    print(f"\n[Verificación] No se pudo convertir 'club' a numérico: {e}")

# 2. Si 'club' es numérico, entonces aplicamos el renombramiento:
if 'club_num' in df_elo_aug.columns and df_elo_aug['club_num'].notna().sum() > len(df_elo_aug) * 0.5:
    print("\n[Corrección] Aplicando renombramiento:")
    # Guardamos el contenido de 'club' (que es ELO) en 'elo_correcto'
    df_elo_aug['elo_correcto'] = df_elo_aug['club']
    # Guardamos el contenido de 'country' (que es club) en 'club_correcto'
    df_elo_aug['club_correcto'] = df_elo_aug['country']
    
    # Eliminamos las columnas originales incorrectas
    df_elo_aug.drop(columns=['country', 'club', 'elo'], inplace=True)
    
    # Renombramos las nuevas columnas
    df_elo_aug.rename(columns={'elo_correcto': 'elo', 'club_correcto': 'club'}, inplace=True)
    
    print("   - 'country' → 'club' (contenido original de country)")
    print("   - 'club' → 'elo' (contenido original de club)")
    print("   - 'elo' original eliminado (basura)")
else:
    print("\n[Advertencia] La columna 'club' no parece contener valores numéricos (ELO). Revisa manualmente.")

# Mostramos el resultado
print("\n[Columnas finales después de corrección]:", df_elo_aug.columns.tolist())
print("\n[Muestra de 10 filas - después de corrección]:")
display(df_elo_aug[['rank', 'club', 'elo']].head(10))

# Eliminamos la columna auxiliar si existe
if 'club_num' in df_elo_aug.columns:
    df_elo_aug.drop(columns=['club_num'], inplace=True)


[Verificación] 'club' es numérico en 630 de 630 filas.

[Corrección] Aplicando renombramiento:
   - 'country' → 'club' (contenido original de country)
   - 'club' → 'elo' (contenido original de club)
   - 'elo' original eliminado (basura)

[Columnas finales después de corrección]: ['rank', 'club_num', 'elo', 'club']

[Muestra de 10 filas - después de corrección]:


,rank,club,elo
0,1,ENG 1 Liverpool,1993
1,2,ENG 2 Arsenal,1993
2,3,FRA 3 Paris SG,1975
3,4,ENG 4 Man City,1960
4,5,ESP 5 Barcelona,1945
5,6,ESP 6 Real Madrid,1936
6,7,ITA 7 Inter,1933
7,8,GER 8 Bayern,1919
8,9,ENG 9 Chelsea,1903
9,10,ENG 10 Aston Villa,1873


In [23]:
# ==========================================
# BLOQUE: EXTRACCIÓN DE PAÍS Y LIMPIEZA DE CLUB
# PARA AMBOS DATAFRAMES ELO
# ==========================================

import re

def extract_country_and_clean_club(df, col_club='club'):
    """
    Extrae el código de país (3 letras) del inicio de la columna 'club',
    lo coloca en una nueva columna 'country', y limpia el nombre del club
    eliminando el código de país y el ranking numérico (si existe).
    """
    # Creamos una copia para no modificar el original
    df_clean = df.copy()
    
    # Patrón: 
    # - 3 letras mayúsculas al inicio (código de país)
    # - opcionalmente un número y espacio (ranking)
    # - el resto es el nombre del club
    pattern = r'^(\w{3})\s+(?:\d+\s+)?(.+)$'
    
    # Aplicamos la extracción fila por fila
    countries = []
    club_names = []
    
    for idx, val in df_clean[col_club].items():
        match = re.match(pattern, str(val))
        if match:
            countries.append(match.group(1))  # Código de país (XXX)
            club_names.append(match.group(2)) # Nombre del club sin código y sin ranking
        else:
            # Si no coincide, lo dejamos como NaN en country y el nombre original
            countries.append(None)
            club_names.append(val)
    
    # Asignamos las nuevas columnas
    df_clean['country'] = countries
    df_clean[col_club] = club_names
    
    # Eliminamos filas donde no se pudo extraer país (opcional, pero recomendado)
    missing_country = df_clean['country'].isna().sum()
    if missing_country > 0:
        print(f"[Advertencia] {missing_country} filas no tienen código de país válido. Se mantienen con country=None.")
    
    return df_clean

# --- Aplicar al ELO 2026-01-06 ---
print("="*60)
print("ELO 2026-01-06: Limpieza de país y club")
print("="*60)
print("\n[Antes de limpieza - muestra de 'club']:")
print(df_elo_jan['club'].head(10).tolist())

df_elo_jan_clean = extract_country_and_clean_club(df_elo_jan)

print("\n[Después de limpieza - muestra de 'club' y 'country']:")
print(df_elo_jan_clean[['club', 'country']].head(10))

# Verificar que todo quedó bien
print(f"\nCantidad de países únicos: {df_elo_jan_clean['country'].nunique()}")
print(f"Top 5 países: {df_elo_jan_clean['country'].value_counts().head(5)}")

# --- Aplicar al ELO 2025-08-02 ---
print("\n" + "="*60)
print("ELO 2025-08-02: Limpieza de país y club")
print("="*60)
print("\n[Antes de limpieza - muestra de 'club']:")
print(df_elo_aug['club'].head(10).tolist())

df_elo_aug_clean = extract_country_and_clean_club(df_elo_aug)

print("\n[Después de limpieza - muestra de 'club' y 'country']:")
print(df_elo_aug_clean[['club', 'country']].head(10))

print(f"\nCantidad de países únicos: {df_elo_aug_clean['country'].nunique()}")
print(f"Top 5 países: {df_elo_aug_clean['country'].value_counts().head(5)}")

# --- Reemplazar los DataFrames originales por los limpios ---
df_elo_jan = df_elo_jan_clean
df_elo_aug = df_elo_aug_clean

print("\n✅ [OK] Ambos DataFrames ELO han sido actualizados con la columna 'country' y nombres de club limpios.")

ELO 2026-01-06: Limpieza de país y club

[Antes de limpieza - muestra de 'club']:
['ENG 1 Arsenal', 'GER 2 Bayern', 'ENG 3 Man City', 'FRA 4 Paris SG', 'ESP 5 Barcelona', 'ENG 6 Liverpool', 'ITA 7 Inter', 'ENG 8 Aston Villa', 'ESP 9 Real Madrid', 'ENG 10 Chelsea']

[Después de limpieza - muestra de 'club' y 'country']:
          club country
0      Arsenal     ENG
1       Bayern     GER
2     Man City     ENG
3     Paris SG     FRA
4    Barcelona     ESP
5    Liverpool     ENG
6        Inter     ITA
7  Aston Villa     ENG
8  Real Madrid     ESP
9      Chelsea     ENG

Cantidad de países únicos: 55
Top 5 países: country
ENG    44
ESP    42
ITA    40
FRA    36
GER    36
Name: count, dtype: int64

ELO 2025-08-02: Limpieza de país y club

[Antes de limpieza - muestra de 'club']:
['ENG 1 Liverpool', 'ENG 2 Arsenal', 'FRA 3 Paris SG', 'ENG 4 Man City', 'ESP 5 Barcelona', 'ESP 6 Real Madrid', 'ITA 7 Inter', 'GER 8 Bayern', 'ENG 9 Chelsea', 'ENG 10 Aston Villa']

[Después de limpieza - muestra

In [24]:
if cols_drop_elo_jan:
    cols_to_drop = [col for col in cols_drop_elo_jan if col in df_elo_jan.columns]
    if cols_to_drop:
        df_elo_jan.drop(columns=cols_to_drop, inplace=True)
        print(f"[ELO 2026-01-06] Columnas eliminadas: {cols_to_drop}")
    else:
        print("[ELO 2026-01-06] Ninguna de las columnas especificadas existe.")

if cols_drop_elo_aug:
    cols_to_drop = [col for col in cols_drop_elo_aug if col in df_elo_aug.columns]
    if cols_to_drop:
        df_elo_aug.drop(columns=cols_to_drop, inplace=True)
        print(f"[ELO 2025-08-02] Columnas eliminadas: {cols_to_drop}")
    else:
        print("[ELO 2025-08-02] Ninguna de las columnas especificadas existe.")

print(df_elo_jan.head())
print(df_elo_aug.head())

[ELO 2026-01-06] Ninguna de las columnas especificadas existe.
[ELO 2025-08-02] Ninguna de las columnas especificadas existe.
   rank       club   elo country
0     1    Arsenal  2054     ENG
1     2     Bayern  1993     GER
2     3   Man City  1993     ENG
3     4   Paris SG  1968     FRA
4     5  Barcelona  1945     ESP
   rank   elo       club country
0     1  1993  Liverpool     ENG
1     2  1993    Arsenal     ENG
2     3  1975   Paris SG     FRA
3     4  1960   Man City     ENG
4     5  1945  Barcelona     ESP


In [25]:
# =============================================
# FILTRADO Y CONTEO DE ELO POR PAÍS
# =============================================

# Lista de países objetivo (en mayúsculas de 3 letras)
target_countries = [
    'ENG', 'ESP', 'FRA', 'GER', 'ITA', 'GRE',  # Europa
    'USA', 'MEX', 'VEN', 'ARG', 'BRA',          # América
    'JPN', 'KOR', 'KSA', 'IRN'                 # Asia
]

# Filtrar ambos DataFrames
elo_jan_filtrado = df_elo_jan[df_elo_jan['country'].isin(target_countries)].copy()
elo_aug_filtrado = df_elo_aug[df_elo_aug['country'].isin(target_countries)].copy()

# Resetear índices
elo_jan_filtrado.reset_index(drop=True, inplace=True)
elo_aug_filtrado.reset_index(drop=True, inplace=True)

# Conteo por país en cada fecha
print("="*60)
print("📊 CONTEO DE EQUIPOS POR PAÍS EN ELO")
print("="*60)

print("\n📌 ELO 2026-01-06:")
print(elo_jan_filtrado['country'].value_counts().sort_index())

print("\n📌 ELO 2025-08-02:")
print(elo_aug_filtrado['country'].value_counts().sort_index())

# Resumen de tamaños
print("\n" + "-"*60)
print(f"Total equipos en ELO Ene (filtrado): {len(elo_jan_filtrado)}")
print(f"Total equipos en ELO Ago (filtrado): {len(elo_aug_filtrado)}")

# (Opcional) Guardar versiones filtradas para usar después
# elo_jan_filtrado.to_parquet("elo_jan_filtrado.parquet")
# elo_aug_filtrado.to_parquet("elo_aug_filtrado.parquet")

📊 CONTEO DE EQUIPOS POR PAÍS EN ELO

📌 ELO 2026-01-06:
country
ENG    44
ESP    42
FRA    36
GER    36
GRE    14
ITA    40
Name: count, dtype: int64

📌 ELO 2025-08-02:
country
ENG    44
ESP    42
FRA    36
GER    36
GRE    14
ITA    40
Name: count, dtype: int64

------------------------------------------------------------
Total equipos en ELO Ene (filtrado): 212
Total equipos en ELO Ago (filtrado): 212


In [26]:
# =============================================
# CARGA DE PDF CONMEBOL (Ranking de Clubes)
# =============================================
import os
import pandas as pd
import numpy as np
import re

# Instalación de librerías (si no las tienes ya)
!pip install pdfplumber tabula-py pypdf2 --quiet

import pdfplumber
import tabula

BASE_DATA_DIR = "/content/drive/MyDrive/TEG02-2025Arellano/Data"
pdf_path = os.path.join(BASE_DATA_DIR, "Ranking-de-Clubes-CONMEBOL-2026-1.pdf")

print("="*60)
print("📄 PROCESANDO PDF: Ranking-de-Clubes-CONMEBOL-2026-1.pdf")
print("="*60)

# --- 1. Intento con tabula-py (más preciso para tablas con bordes) ---
try:
    tables = tabula.read_pdf(
        pdf_path,
        pages='all',           # Todas las páginas
        multiple_tables=True,  # Cada página es una tabla independiente
        guess=True,            # Auto-detecta área de tabla
        pandas_options={'header': None}  # No asumimos header automático
    )
    print(f"[tabula-py] Extraídas {len(tables)} tablas (páginas).")
    
    # Unir todas las tablas verticalmente
    df_conmebol_raw = pd.concat(tables, ignore_index=True)
    print(f"[OK] Shape inicial (sin procesar): {df_conmebol_raw.shape}")
    
except Exception as e:
    print(f"[WARN] tabula-py falló: {e}. Recurriendo a pdfplumber...")
    all_rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            page_table = page.extract_table()
            if page_table:
                # Limpiar celdas vacías y saltos de línea
                clean_page = [[str(cell).replace('\n', ' ').strip() if cell else '' for cell in row] for row in page_table]
                all_rows.extend(clean_page)
    df_conmebol_raw = pd.DataFrame(all_rows)
    print(f"[OK] pdfplumber extrajo {len(df_conmebol_raw)} filas.")

# --- 2. Mostrar primeras filas para entender la estructura ---
print("\n[Vista previa de primeras 10 filas (RAW)]:")
display(df_conmebol_raw.head(10))

📄 PROCESANDO PDF: Ranking-de-Clubes-CONMEBOL-2026-1.pdf
[WARN] tabula-py falló: Expecting ',' delimiter: line 1 column 7 (char 6). Recurriendo a pdfplumber...
[OK] pdfplumber extrajo 309 filas.

[Vista previa de primeras 10 filas (RAW)]:


,0,1,2,3,4,5
0,Posición,Equipo,País,Histórico (Libertadores + Sudamericana),Performance (Libertadores + Sudamericana),Total
1,1,Palmeiras,Brasil,"1157,6",8671,"9828,6"
2,2,Flamengo,Brasil,694,"8609,1","9303,1"
3,3,River Plate,Argentina,"2626,8",6136,"8762,8"
4,4,Boca Juniors,Argentina,"2785,2","4574,8",7360
5,5,Peñarol,Uruguay,"2864,4",3165,"6029,4"
6,6,Atlético Mineiro,Brasil,"447,2",5502,"5949,2"
7,7,São Paulo,Brasil,"1764,8","3866,7","5631,5"
8,8,Nacional,Uruguay,"2420,4","3100,6",5521
9,9,Racing Club,Argentina,"522,8","4789,9","5312,7"


In [27]:
# =============================================
# LIMPIEZA FINAL CONMEBOL: DROP DE FILAS Y COLUMNAS
# =============================================

# 1. Verificar nombres de columnas actuales
print("📋 Columnas actuales en df_conmebol_raw:")
print(df_conmebol_raw.columns.tolist())
df_conmebol=df_conmebol_raw
# =============================================
# FIX: ASIGNAR PRIMERA FILA COMO ENCABEZADO
# (Insertar entre Paso 1 y Paso 2 del bloque anterior)
# =============================================
df = df_conmebol  # cambia según tu variable real

# 1. Obtener valores de la primera fila como strings en minúsculas
first_row_vals = df.iloc[0].astype(str).str.lower().str.strip().tolist()

# 2. Palabras clave que indican que la fila es un encabezado
keywords = ['pos', 'equipo', 'pais', 'puntaje', 'total', 'rank', 'team', 'country', 'points', 'club']

# 3. Verificar si al menos una palabra clave aparece en la primera fila
is_header = any(any(kw in val for kw in keywords) for val in first_row_vals)

if is_header:
    print("[Fix] Detectada primera fila con nombres de columnas. Aplicando fix...")
    # Tomar primera fila como nuevos nombres
    new_headers = df.iloc[0].astype(str).str.strip()
    df.columns = new_headers
    # Eliminar la primera fila (datos)
    df = df.iloc[1:].reset_index(drop=True)
    # Estandarizar nombres a snake_case (minúsculas y guiones bajos)
    import re
    def standardize(col):
        col = str(col).lower().strip()
        col = re.sub(r'[^a-z0-9]', '_', col)
        col = re.sub(r'_+', '_', col)
        return col
    df.columns = [standardize(c) for c in df.columns]
    print("[Fix] ✅ Aplicado correctamente. Nuevas columnas:", df.columns.tolist())
else:
    print("[Fix] La primera fila NO parece ser encabezado. Se asume que el DataFrame ya tiene nombres de columnas correctos.")
    print("   Columnas actuales:", df.columns.tolist())

# 4. Asignar de vuelta a la variable original (si usaste df)
df_conmebol_filtrado = df

# 5. Verificar resultado
print("\n[Vista previa después del fix (primeras 3 filas)]:")
display(df_conmebol_filtrado.head(3))


# 4. Estandarizamos los nombres de columna (a minúsculas y snake_case)
def standardize(col):
    import re
    col = str(col).lower().strip()
    col = re.sub(r'[^a-z0-9]', '_', col)
    col = re.sub(r'_+', '_', col)
    return col

df_conmebol_filtrado.columns = [standardize(c) for c in df_conmebol_filtrado.columns]

print("   Nombres estandarizados a:", df_conmebol.columns.tolist())

# 5. Verificamos que haya quedado bien
print("\n[Vista previa después del fix]:")
display(df_conmebol_filtrado.head())

# 2. Eliminar filas donde 'Total' sea 0 (o NaN)
# Asumimos que la columna se llama 'total' (estandarizada a minúsculas)
# Si se llama 'points' o 'puntaje', ajustamos.
# Primero identificamos la columna de puntaje total
total_col = None
for col in df_conmebol_filtrado.columns:
    if 'total' in col.lower() or 'puntaje' in col.lower() or 'points' in col.lower():
        total_col = col
        break

if total_col:
    print(f"\n🔍 Columna de puntaje total detectada: '{total_col}'")
    # Convertir a numérico (por si hay comas)
    df_conmebol_filtrado[total_col] = pd.to_numeric(
        df_conmebol_filtrado[total_col].astype(str).str.replace(',', '.'),
        errors='coerce'
    )
    # Contar cuántas filas tienen 0 o NaN
    zero_rows = (df_conmebol_filtrado[total_col] == 0) | (df_conmebol_filtrado[total_col].isna())
    print(f"   Filas con Total = 0 o NaN: {zero_rows.sum()}")
    # Eliminarlas
    df_conmebol_filtrado = df_conmebol_filtrado[~zero_rows].copy()
    print(f"   Shape después de eliminar filas: {df_conmebol_filtrado.shape}")
else:
    print("⚠️ No se encontró columna de puntaje total. Saltando eliminación de filas.")

# 3. Eliminar columnas específicas por nombre (exacto o aproximado)
cols_to_drop = []
for col in df_conmebol_filtrado.columns:
    if 'hist_rico' in col.lower() and 'libertadores' in col.lower():
        cols_to_drop.append(col)
    if 'performance' in col.lower() and 'libertadores' in col.lower():
        cols_to_drop.append(col)

if cols_to_drop:
    print(f"\n🗑️ Columnas a eliminar: {cols_to_drop}")
    df_conmebol_filtrado.drop(columns=cols_to_drop, inplace=True)
    print("   Columnas eliminadas.")
else:
    print("\n⚠️ No se encontraron las columnas 'Histórico...' o 'Performance...' para eliminar.")

# 4. Resetear índice
df_conmebol_filtrado.reset_index(drop=True, inplace=True)

# 5. Mostrar resultado final
print("\n" + "="*60)
print("📊 CONMEBOL - RESULTADO FINAL")
print("="*60)
print(f"Total de equipos: {len(df_conmebol_filtrado)}")
print("\nDistribución por país:")
print(df_conmebol_filtrado['pa_s'].value_counts().sort_index())

print("\n[Vista previa de las primeras 5 filas finales]:")
display(df_conmebol_filtrado.head())

# Opcional: guardar versión final
# df_conmebol_raw.to_parquet(os.path.join(BASE_DATA_DIR, "conmebol_final.parquet"))

📋 Columnas actuales en df_conmebol_raw:
[0, 1, 2, 3, 4, 5]
[Fix] Detectada primera fila con nombres de columnas. Aplicando fix...
[Fix] ✅ Aplicado correctamente. Nuevas columnas: ['posici_n', 'equipo', 'pa_s', 'hist_rico_libertadores_sudamericana_', 'performance_libertadores_sudamericana_', 'total']

[Vista previa después del fix (primeras 3 filas)]:


,posici_n,equipo,pa_s,hist_rico_libertadores_sudamericana_,performance_libertadores_sudamericana_,total
0,1,Palmeiras,Brasil,"1157,6",8671,"9828,6"
1,2,Flamengo,Brasil,694,"8609,1","9303,1"
2,3,River Plate,Argentina,"2626,8",6136,"8762,8"


   Nombres estandarizados a: ['Posición', 'Equipo', 'País', 'Histórico (Libertadores + Sudamericana)', 'Performance (Libertadores + Sudamericana)', 'Total']

[Vista previa después del fix]:


,posici_n,equipo,pa_s,hist_rico_libertadores_sudamericana_,performance_libertadores_sudamericana_,total
0,1,Palmeiras,Brasil,"1157,6",8671,"9828,6"
1,2,Flamengo,Brasil,694,"8609,1","9303,1"
2,3,River Plate,Argentina,"2626,8",6136,"8762,8"
3,4,Boca Juniors,Argentina,"2785,2","4574,8",7360
4,5,Peñarol,Uruguay,"2864,4",3165,"6029,4"



🔍 Columna de puntaje total detectada: 'total'
   Filas con Total = 0 o NaN: 22
   Shape después de eliminar filas: (286, 6)

🗑️ Columnas a eliminar: ['hist_rico_libertadores_sudamericana_', 'performance_libertadores_sudamericana_']
   Columnas eliminadas.

📊 CONMEBOL - RESULTADO FINAL
Total de equipos: 286

Distribución por país:
pa_s
Argentina        28
Bolivia          26
Brasil           40
Chile            24
Colombia         21
Ecuador          23
Mexico           17
Paraguay         20
Perú             32
United States     1
Uruguay          20
Venezuela        34
Name: count, dtype: int64

[Vista previa de las primeras 5 filas finales]:


,posici_n,equipo,pa_s,total
0,1,Palmeiras,Brasil,9828.6
1,2,Flamengo,Brasil,9303.1
2,3,River Plate,Argentina,8762.8
3,4,Boca Juniors,Argentina,7360.0
4,5,Peñarol,Uruguay,6029.4


In [28]:
# =============================================
# DETECCIÓN DE PAÍSES ÚNICOS EN CONMEBOL
# =============================================

# 1. Identificar la columna de país (puede llamarse 'pais', 'país', 'country', o 'pa_s')
country_col = None
for col in df_conmebol_filtrado.columns:
    col_lower = col.lower()
    if 'pa_s' in col_lower or 'país' in col_lower or 'country' in col_lower or 'nation' in col_lower:
        country_col = col
        break

if country_col is None:
    # Si no encuentra ninguna, buscar por la primera columna que tenga strings y parezca categórica
    for col in df_conmebol_filtrado.columns:
        if df_conmebol_filtrado[col].dtype == 'object':
            # Revisar si tiene pocos valores únicos (sospecha de país)
            n_unique = df_conmebol_filtrado[col].nunique()
            if 2 < n_unique < 20:  # Rango típico para países en CONMEBOL (10-12 países)
                country_col = col
                break

if country_col:
    print(f"🔍 Columna de país detectada: '{country_col}'")
    # Obtener valores únicos (limpiando espacios y mayúsculas)
    unique_countries = df_conmebol_filtrado[country_col].astype(str).str.strip().unique()
    print("\n🌍 Países únicos en CONMEBOL (valores crudos):")
    for c in sorted(unique_countries):
        print(f"  - '{c}'")
    
    # Conteo por país
    print("\n📊 Conteo de equipos por país:")
    print(df_conmebol_filtrado[country_col].value_counts().sort_index())
    
else:
    print("⚠️ No se pudo identificar la columna de país. Mostrando todas las columnas:")
    print(df_conmebol_filtrado.columns.tolist())
    print("\nPrimeras filas para inspección manual:")
    display(df_conmebol_filtrado.head())

🔍 Columna de país detectada: 'pa_s'

🌍 Países únicos en CONMEBOL (valores crudos):
  - 'Argentina'
  - 'Bolivia'
  - 'Brasil'
  - 'Chile'
  - 'Colombia'
  - 'Ecuador'
  - 'Mexico'
  - 'Paraguay'
  - 'Perú'
  - 'United States'
  - 'Uruguay'
  - 'Venezuela'

📊 Conteo de equipos por país:
pa_s
Argentina        28
Bolivia          26
Brasil           40
Chile            24
Colombia         21
Ecuador          23
Mexico           17
Paraguay         20
Perú             32
United States     1
Uruguay          20
Venezuela        34
Name: count, dtype: int64


In [29]:
# =============================================
# FILTRADO Y MAPEO DE PAÍSES CONMEBOL
# =============================================

# 1. Limpiar espacios en blanco en la columna 'pa_s' (o como se llame)
# Primero identificamos el nombre correcto de la columna de país
country_col = None
for col in df_conmebol_filtrado.columns:
    if 'pais' in col or 'pa_s' in col or 'country' in col:
        country_col = col
        break

if country_col is None:
    raise ValueError("No se encontró la columna de país. Verifica los nombres.")

print(f"🔍 Columna de país detectada: '{country_col}'")

# Limpiar espacios y convertir a string
df_conmebol_filtrado[country_col] = df_conmebol_filtrado[country_col].astype(str).str.strip()

# 2. Definir países objetivo (en el formato que aparecen en el DataFrame)
# Usamos los nombres exactos que viste en la lista de únicos
target_countries_raw = ['Argentina', 'Brasil', 'Venezuela']

# 3. Aplicar filtro
df_conmebol_filtrado = df_conmebol_filtrado[df_conmebol_filtrado[country_col].isin(target_countries_raw)].copy()

print(f"\n✅ Filtrando por {len(target_countries_raw)} países: {target_countries_raw}")
print(f"   Shape después del filtro: {df_conmebol_filtrado.shape}")

# 4. Mapear nombres completos a códigos de 3 letras (ISO)
country_mapping = {
    'Argentina': 'ARG',
    'Brasil': 'BRA',
    'Venezuela': 'VEN'
}

df_conmebol_filtrado['country_code'] = df_conmebol_filtrado[country_col].map(country_mapping)

# 5. Reemplazar la columna original por el código (o crear una nueva)
# Recomiendo mantener ambas: la original con el nombre completo y una nueva con el código.
# Pero para consistencia con los otros dataframes, usaremos 'country' como código.
df_conmebol_filtrado['country'] = df_conmebol_filtrado['country_code']

# Opcional: eliminar la columna original si no la necesitas
# df_conmebol_filtrado.drop(columns=[country_col], inplace=True)

# 6. Resetear índice
df_conmebol_filtrado.reset_index(drop=True, inplace=True)

# 7. Reporte final
print("\n" + "="*60)
print("📊 CONMEBOL - FILTRADO FINAL POR PAÍS")
print("="*60)
print("\nDistribución por país (código):")
print(df_conmebol_filtrado['country'].value_counts().sort_index())

print(f"\nTotal de equipos en CONMEBOL (ARG, BRA, VEN): {len(df_conmebol_filtrado)}")

# 8. Vista previa para confirmar
print("\n[Vista previa de las primeras 5 filas con países mapeados]:")
display(df_conmebol_filtrado[[country_col, 'equipo', 'country']].head())

🔍 Columna de país detectada: 'pa_s'

✅ Filtrando por 3 países: ['Argentina', 'Brasil', 'Venezuela']
   Shape después del filtro: (102, 4)

📊 CONMEBOL - FILTRADO FINAL POR PAÍS

Distribución por país (código):
country
ARG    28
BRA    40
VEN    34
Name: count, dtype: int64

Total de equipos en CONMEBOL (ARG, BRA, VEN): 102

[Vista previa de las primeras 5 filas con países mapeados]:


,pa_s,equipo,country
0,Brasil,Palmeiras,BRA
1,Brasil,Flamengo,BRA
2,Argentina,River Plate,ARG
3,Argentina,Boca Juniors,ARG
4,Brasil,Atlético Mineiro,BRA


In [30]:
# =============================================
# FILTRADO Y RENOMBRAMIENTO A INGLÉS (CONMEBOL)
# =============================================

# 1. Filtrar por los 3 países objetivo de Sudamérica
target_conmebol = ['Argentina', 'Brasil', 'Venezuela']
df_conmebol_filtrado = df_conmebol_filtrado[df_conmebol_filtrado['pa_s'].isin(target_conmebol)].copy()

print("✅ [CONMEBOL] Filtrado aplicado.")
print(df_conmebol_filtrado['pa_s'].value_counts().sort_index())

# 2. Renombrar columnas de español a inglés
# Asumo que las columnas actuales son: 'pos', 'equipo', 'pa_s', y la columna de puntaje total.
# Primero veamos cuáles son exactamente
print("\n📋 Columnas actuales antes del renombramiento:")
print(df_conmebol_filtrado.columns.tolist())

# Diccionario de mapeo (ajusta según los nombres exactos que veas)
rename_map = {
    'pos': 'rank',                # Posición
    'equipo': 'team',              # Equipo/Club
    'pa_s': 'country',             # País
    'puntaje_total': 'total_points', # Puntaje total (si se llama así)
    'total': 'total_points',       # Alternativa
    'puntos': 'total_points'       # Alternativa
}

# Aplicar solo las columnas que existen
columns_to_rename = {k: v for k, v in rename_map.items() if k in df_conmebol_filtrado.columns}
df_conmebol_filtrado.rename(columns=columns_to_rename, inplace=True)

print("\n✅ Columnas renombradas a inglés:")
print(df_conmebol_filtrado.columns.tolist())

# 3. Resetear índice
df_conmebol_filtrado.reset_index(drop=True, inplace=True)

# 4. Mostrar resultado final
print("\n" + "="*60)
print("📊 CONMEBOL - VERSIÓN FINAL (EN INGLÉS)")
print("="*60)
print(f"Total de equipos: {len(df_conmebol_filtrado)}")
print("\nDistribución por país:")
print(df_conmebol_filtrado['country'].value_counts().sort_index())

print("\n[Vista previa de las primeras 5 filas]:")
display(df_conmebol_filtrado.head())

# (Opcional) Guardar versión final
# df_conmebol_filtrado.to_parquet(os.path.join(BASE_DATA_DIR, "conmebol_final.parquet"))

✅ [CONMEBOL] Filtrado aplicado.
pa_s
Argentina    28
Brasil       40
Venezuela    34
Name: count, dtype: int64

📋 Columnas actuales antes del renombramiento:
['posici_n', 'equipo', 'pa_s', 'total', 'country_code', 'country']

✅ Columnas renombradas a inglés:
['posici_n', 'team', 'country', 'total_points', 'country_code', 'country']

📊 CONMEBOL - VERSIÓN FINAL (EN INGLÉS)
Total de equipos: 102

Distribución por país:


ValueError: Grouper for 'country' not 1-dimensional

In [ ]:
# =============================================
# DIAGNÓSTICO ESTRUCTURAL DE TODOS LOS DATAFRAMES
# =============================================

dataframes_dict = {
    'UEFA': df_uefa,
    'AFC': df_afc,
    'ELO_Ene': df_elo_jan_clean,
    'ELO_Ago': df_elo_aug_clean,
    'CONMEBOL': df_conmebol_filtrado
}

print("="*60)
print("🔍 DIAGNÓSTICO DE COLUMNAS Y TIPOS")
print("="*60)

for name, df in dataframes_dict.items():
    print(f"\n📌 {name}")
    print(f"   Shape: {df.shape}")
    print(f"   Columnas: {df.columns.tolist()}")
    print(f"   Tipos de columnas: {df.dtypes}")
    # Mostrar primeras 2 filas
    print("   Primeras 2 filas:")
    display(df.head(2))

In [ ]:
# =============================================
# VERIFICAR PAÍSES EN ELO
# =============================================
print("🌍 Países únicos en ELO Enero (sin filtrar):")
print(sorted(df_elo_jan['country'].unique()))

print("\n🌍 Países únicos en ELO Agosto (sin filtrar):")
print(sorted(df_elo_aug['country'].unique()))

In [ ]:
# =============================================
# FILTRAR ELO POR PAÍSES OBJETIVO
# =============================================

# Definir los países objetivo (los mismos de siempre)
target_countries = [
    'ENG', 'ESP', 'FRA', 'GER', 'ITA', 'GRE',  # Europa
    'USA', 'MEX', 'VEN', 'ARG', 'BRA',          # América
    'JPN', 'KOR', 'KSA', 'IRN'                 # Asia
]

# Aplicar filtro a ambos ELO (si no se hizo ya)
df_elo_jan = df_elo_jan[df_elo_jan['country'].isin(target_countries)].copy()
df_elo_aug = df_elo_aug[df_elo_aug['country'].isin(target_countries)].copy()

print(f"✅ ELO Enero filtrado: {len(df_elo_jan)} equipos")
print(f"✅ ELO Agosto filtrado: {len(df_elo_aug)} equipos")

# Resetear índices
df_elo_jan.reset_index(drop=True, inplace=True)
df_elo_aug.reset_index(drop=True, inplace=True)

In [ ]:
# =============================================
# VERIFICAR QUE ELO ESTÁ FILTRADO POR PAÍSES
# =============================================

print("🌍 Países únicos en ELO Enero (después del filtro):")
print(sorted(df_elo_jan['country'].unique()))

print("\n🌍 Países únicos en ELO Agosto (después del filtro):")
print(sorted(df_elo_aug['country'].unique()))

In [ ]:
# =============================================
# EXTACCIÓN DE EQUIPOS ÚNICOS (CON LIMPIEZA DE NaN)
# =============================================

import numpy as np

# Unir todos los equipos de todos los DataFrames (solo los que tienen columna de equipo)
all_teams = set()

# Función auxiliar para agregar equipos de un DataFrame
def add_teams(df, col_name):
    if col_name in df.columns:
        # Convertir a string, eliminar NaN y valores vacíos
        teams = df[col_name].dropna().astype(str).str.strip()
        teams = teams[teams != '']
        all_teams.update(teams)

# Agregar de cada DataFrame
add_teams(df_uefa_filtrado, 'team')
add_teams(df_afc_filtrado, 'team')
add_teams(df_elo_jan, 'club')        # ELO Enero usa 'club'
add_teams(df_elo_aug, 'club')        # ELO Agosto usa 'club'
add_teams(df_conmebol_filtrado, 'team')

# Filtrar para eliminar cualquier valor que no sea string (por si quedó algún float)
all_teams = {str(t) for t in all_teams if isinstance(t, str) or not pd.isna(t)}

# Ordenar alfabéticamente (ahora todos son strings)
all_teams_sorted = sorted(all_teams)

print(f"📊 Total de equipos únicos (filtrando por países objetivo): {len(all_teams_sorted)}")
print("\n📋 Muestra (primeros 50 equipos):")
print(all_teams_sorted[:50])

In [ ]:
# =============================================
# UNIFICACIÓN DE COLUMNAS Y EXTRACCIÓN DE ÚNICOS
# =============================================

# 1. Diccionario con los DataFrames (usando tus nombres)
dfs = {
    'UEFA': df_uefa,
    'AFC': df_afc,
    'ELO_Ene': df_elo_jan,
    'ELO_Ago': df_elo_aug,
    'CONMEBOL': df_conmebol_filtrado
}

# 2. Mapeo de renombres (de nombres actuales a nombres estándar)
rename_map = {
    # Para equipo
    'Team': 'team',
    'club': 'team',
    'equipo': 'team',
    # Para país
    'Country': 'country',
    'pais': 'country',
    'pa_s': 'country',
    'country_code': 'country',  # si existe y se llama así
    # Para puntos/elo
    'puntos_uefa': 'points',
    'Points': 'points',
    'elo': 'elo',  # lo dejamos como está
    'total_points': 'total_points',  # específico de CONMEBOL
    # Para rank
    'Rank': 'rank',
    'rank': 'rank',
    'posici_n': 'rank',  # CONMEBOL
    'pos': 'rank',
    # Otros
    'Rank_local': 'rank_local',  # si lo quieres conservar, lo dejamos
    'pais_equipo': 'country_coefficient',  # renombramos para que no confunda
}

# 3. Aplicar renombres y eliminar duplicados
for name, df in dfs.items():
    # Renombrar columnas que existan
    cols_to_rename = {old: new for old, new in rename_map.items() if old in df.columns}
    if cols_to_rename:
        df.rename(columns=cols_to_rename, inplace=True)
        print(f"✅ {name}: renombradas {list(cols_to_rename.values())}")
    
    # Eliminar columnas duplicadas (CONMEBOL tiene 'country' duplicado)
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Asegurar que los nombres estén en minúsculas (ya casi todos)
    df.columns = [c.lower() for c in df.columns]
    
    # Asignar de vuelta
    dfs[name] = df

# Actualizar variables originales
df_uefa_filtrado = dfs['UEFA']
df_afc_filtrado = dfs['AFC']
df_elo_jan_filtrado = dfs['ELO_Ene']
df_elo_aug_filtrado = dfs['ELO_Ago']
df_conmebol_filtrado = dfs['CONMEBOL']

# 4. Mostrar columnas finales de cada uno
print("\n" + "="*60)
print("📋 COLUMNAS FINALES DESPUÉS DE UNIFICACIÓN")
print("="*60)
for name, df in dfs.items():
    print(f"\n🔹 {name}: {df.columns.tolist()}")

# 5. Extraer valores únicos de 'team' para ver variantes
print("\n" + "="*60)
print("📊 EQUIPOS ÚNICOS POR DATAFRAME (sin normalizar)")
print("="*60)

# Recopilamos todos los equipos únicos de todos los DataFrames
all_teams = set()
for name, df in dfs.items():
    if 'team' in df.columns:
        unique_teams = sorted(df['team'].astype(str).unique())
        print(f"\n🔹 {name}: {len(unique_teams)} equipos únicos.")
        print(f"   Muestra (primeros 20): {unique_teams[:20]}")
        all_teams.update(unique_teams)
    else:
        print(f"\n⚠️ {name}: No tiene columna 'team'. Columnas: {df.columns.tolist()}")

# (Opcional) Guardar lista completa de equipos para construir el mapeo
all_teams_sorted = sorted(all_teams)
print(f"\n📌 TOTAL DE EQUIPOS ÚNICOS EN TODOS LOS DATAFRAMES: {len(all_teams_sorted)}")
print("   (Muestra de los primeros 30):")
print(all_teams_sorted[:30])

In [ ]:
file_team_review = os.path.join(BASE_DATA_DIR, "team_name_review.csv")

df_team_names = pd.read_csv(file_team_review, encoding='utf-8-sig')  # asumimos UTF-8 con BOM
print("✅ CSV cargado. Shape:", df_team_names.shape)
display(df_team_names.head())

# =============================================
# CONSTRUCCIÓN DEL DICCIONARIO DE MAPEO
# =============================================

# Suponemos que la columna 'Normalized' existe y es la última o una de las columnas.
# Vamos a identificar la columna 'Normalized' y las columnas de nombres alternativos.

# Primero, veamos los nombres de columnas exactos
print("Columnas del CSV:", df_team_names.columns.tolist())

# Identificar la columna 'Normalized' (puede llamarse 'Normalized' o 'normalizado' etc.)
normalized_col = None
for col in df_team_names.columns:
    if 'normalized' in col.lower():
        normalized_col = col
        break

if normalized_col is None:
    raise ValueError("No se encontró columna 'Normalized'. Verifica el CSV.")

# Construir diccionario: por cada fila, mapear todos los nombres alternativos (columnas 2-6) al nombre normalizado.
# Asumimos que las columnas 2-6 son las que contienen nombres alternativos.
# Identificamos estas columnas por su índice o por su nombre (si son fijas).

# Opción 1: Usar índices (asumiendo que las columnas 2 a 6 son las alternativas)
# Si el CSV tiene más de 6 columnas, podríamos elegir las que no son 'Normalized' y no son índice.
# Vamos a tomar todas las columnas excepto la primera (índice 0) y la columna 'Normalized'.
alt_cols = [col for col in df_team_names.columns if col != df_team_names.columns[0] and col != normalized_col]

print(f"Columnas alternativas identificadas: {alt_cols}")

# Construir el diccionario de mapeo
mapping_dict = {}
for idx, row in df_team_names.iterrows():
    normalized_name = row[normalized_col]
    # Para cada columna alternativa, si el valor no es nulo, agregamos al mapeo
    for col in alt_cols:
        alt_name = row[col]
        if pd.notna(alt_name) and isinstance(alt_name, str):
            alt_name_clean = alt_name.strip()
            if alt_name_clean:
                mapping_dict[alt_name_clean] = normalized_name

print(f"✅ Diccionario construido con {len(mapping_dict)} entradas.")
# Mostrar algunas muestras
print("\nMuestra del diccionario:")
sample_items = list(mapping_dict.items())[:10]
for k, v in sample_items:
    print(f"  '{k}' → '{v}'")

In [ ]:
# =============================================
# APLICACIÓN DEL MAPEO A TODOS LOS DATAFRAMES
# =============================================

def apply_team_mapping(df, mapping_dict, team_col='team'):
    """
    Aplica el mapeo a la columna 'team' del DataFrame.
    Retorna el DataFrame con la columna mapeada y una lista de valores no encontrados.
    """
    if team_col not in df.columns:
        print(f"⚠️ No se encontró la columna '{team_col}' en el DataFrame.")
        return df, []
    
    # Crear una copia para no modificar el original
    df_copy = df.copy()
    
    # Obtener los valores únicos originales
    original_teams = df_copy[team_col].astype(str).str.strip().unique()
    
    # Mapear los valores
    mapped_teams = []
    not_found = []
    
    for team in original_teams:
        if team in mapping_dict:
            mapped_teams.append(team)
        else:
            not_found.append(team)
    
    # Aplicar el mapeo
    df_copy[team_col] = df_copy[team_col].astype(str).str.strip().map(mapping_dict)
    
    # Los valores no mapeados quedan como NaN (los dejamos para que los veas)
    print(f"   - Equipos mapeados: {len(original_teams) - len(not_found)}")
    print(f"   - Equipos NO encontrados: {len(not_found)}")
    
    return df_copy, not_found

# Aplicar a cada DataFrame
dataframes_dict = {
    'UEFA': df_uefa_filtrado,
    'AFC': df_afc_filtrado,
    'ELO_Ene': df_elo_jan_filtrado,
    'ELO_Ago': df_elo_aug_filtrado,
    'CONMEBOL': df_conmebol_filtrado
}

not_found_all = {}

for name, df in dataframes_dict.items():
    print(f"\n🔹 {name}:")
    df_mapped, not_found = apply_team_mapping(df, mapping_dict, team_col='team')
    dataframes_dict[name] = df_mapped
    if not_found:
        not_found_all[name] = not_found
        print(f"   ⚠️ Equipos no encontrados en {name}: {not_found[:10]}...")  # muestra primeros 10

# Actualizar variables globales
df_uefa_filtrado = dataframes_dict['UEFA']
df_afc_filtrado = dataframes_dict['AFC']
df_elo_jan_filtrado = dataframes_dict['ELO_Ene']
df_elo_aug_filtrado = dataframes_dict['ELO_Ago']
df_conmebol_filtrado = dataframes_dict['CONMEBOL']

In [ ]:
# =============================================
# LISTA DE EQUIPOS NO ENCONTRADOS (PARA REVISIÓN MANUAL)
# =============================================

# Unir todos los no encontrados en un solo set
all_not_found = set()
for name, not_found_list in not_found_all.items():
    all_not_found.update(not_found_list)

print("\n" + "="*60)
print("📋 LISTA COMPLETA DE EQUIPOS NO ENCONTRADOS (para revisión manual)")
print("="*60)
print(f"Total de equipos únicos no encontrados: {len(all_not_found)}")
print("\nLista ordenada:")
for team in sorted(all_not_found):
    print(f"  - '{team}'")

# Exportar a un CSV para que puedas revisarlo y agregar los mapeos manualmente
df_not_found = pd.DataFrame(sorted(all_not_found), columns=['team_name'])
df_not_found.to_csv(os.path.join(BASE_DATA_DIR, "not_found_teams.csv"), index=False)
print(f"\n✅ Lista exportada a '{BASE_DATA_DIR}/not_found_teams.csv'")

In [ ]:
# =============================================
# FILTRAR EQUIPOS #drop DE ELO AGOSTO
# =============================================

# 1. Definir el diccionario manual (el que construiste)
manual_mapping = {
    # Bloque Numéricos
    '1.FC Heidenheim': 'Heidenheim',
    '1.FC KÃ¶ln': 'Koln',
    '1899 Hoffenheim': 'Hoffenheim',
    
    # Bloque A
    'aek athens':'AEK',
    'aek':'AEK',
    'AS Monaco': 'Monaco',
    'Alavés': 'Alaves',
    'Albacete': 'Albacete',  # drop
    'Almería': 'Almeria',    # drop
    'Andorra CF': 'Andorra', # drop
    'Annecy': 'Annecy',      # drop
    'Aris Thessaloniki': 'Aris',
    'Aston Villa': 'Villa',
    'Athletic Bilbao': 'AthBilbao',
    'Atlético': 'AtlMadrid',
    'Avellino': 'Avellino',  # drop
    
    # Bloque B
    'Bari': 'Bari',          # drop
    'Bastia': 'Bastia',      # drop
    'Bayer Leverkusen': 'Leverkusen',
    'Bayern MÃ¼nchen': 'Bayern',
    'Bielefeld': 'Arminia',
    'Bilbao': 'AthBilbao',
    'Birmingham': 'Birmingham', # drop
    'Blackburn': 'Blackburn',   # drop
    'Borussia Dortmund': 'Dortmund',
    'Borussia MÃ¶nchengladbach': 'MGladbach',
    'Boulogne': 'Bologna', 
    'Braunschweig': 'Braunschweig', # drop
    'Brighton & Hove Albion': 'Brighton',
    'Bristol City': 'Bristol City', # drop
    'Burgos': 'Burgos',             # drop
    
    # Bloque C
    'Carrarese': 'Carrarese',       # drop
    'Cartagena': 'Cartagena',       # drop
    'Castellón': 'Castellon',       # drop
    'Catanzaro': 'Catanzaro',       # drop
    'Cesena': 'Cesena',             # drop
    'Ceuta': 'Ceuta',               # drop
    'Charlton': 'Charlton',         # drop
    'Cittadella': 'Cittadella',     # drop
    'Clermont': 'Clermont63',
    'Cosenza': 'Cosenza',           # drop
    'Coventry': 'Coventry',         # drop
    'Crystal Palace': 'CrystalPalace',
    'Córdoba': 'Cordoba',           # drop
    
    # Bloque D
    'Darmstadt': 'Darmstadt98',
    'Depor': 'Deportivo La Coruna',
    'Derby': 'Derby',               # drop
    'Dresden': 'Dresden',           # drop
    'Dunkerque': 'Dunkerque',       # drop
    'Düsseldorf': 'Dusseldorf',
    
    # Bloque E
    'Eldense': 'Eldense',           # drop
    'Elversberg': 'Elversberg',     # drop
    'Entella': 'Entella',           # drop
    
    # Bloque F
    'FC Barcelona': 'Barcelona',
    'FC Nantes': 'Nantes',
    'Ferrol': 'Ferrol',             # drop
    'Forest': 'NottsForest',
    'Frankfurt': 'Eintracht',
    'Fürth': 'Greuther',
    
    # Bloque G
    'Gijón': 'Gijon',               # drop
    'Girona FC': 'Girona',
    'Granada CF': 'Granada',
    'Grenoble': 'Grenoble',         # drop
    
    # Bloque H
    'Holstein': 'HolsteinKiel',
    'Hull': 'Hull',                 # drop
    
    # Bloque J
    'JeounbukHyundai': 'JeonbukHyundai',
    'Juve Stabia': 'Juve Stabia',   # drop
    'Jwangju': 'Gwangju',
    
    # Bloque K
    'Karlsruhe': 'Karlsruhe',       # drop
    'KawasakiF': 'KawasakiFrontale',
    'Kifisias': 'Kifisia',
    'Köln': 'Koln',
    
    # Bloque L
    'Lautern': 'Lautern',           # drop
    'Laval': 'Laval',               # drop
    'Le Mans': 'Le Mans',           # drop
    'Leeds': 'LeedsUtd',
    'Leicester City': 'Leicester',
    'Leonesa': 'Leonesa',           # drop
    'Lille OSC': 'Lille',
    
    # Bloque M
    'Magdeburg': 'Magdeburg',       # drop
    'Man City': 'ManCity',
    'Man United': 'ManUnited',
    'Manchester City': 'ManCity',
    'Manchester United': 'ManUnited',
    'Mantova': 'Mantova',           # drop
    'Martigues': 'Martigues',       # drop
    'Middlesbrough': 'Middlesbrough', # drop
    'Millwall': 'Millwall',         # drop
    'Mirandes': 'Mirandes',         # drop
    'Modena': 'Modena',             # drop
    'Málaga': 'Malaga',
    'Münster': 'Munster',           # drop
    
    # Bloque N
    'Nancy': 'Nancy',               # drop
    'Newcastle United': 'Newcastle',
    'Nürnberg': 'Nuremberg',
    
    # Bloque O
    'OFI Heraklion': 'OFI',
    'OGC Nice': 'Nice',
    'Olympiakos Piraeus': 'Olympiacos',
    'Olympique Lyon': 'OLyon',
    'Olympique Marseille': 'OMarseille',
    'Oxford': 'Oxford',             # drop
    
    # Bloque P
    'PAOK Thessaloniki': 'PAOK',
    'Padova': 'Padova',             # drop
    'Palermo': 'Palermo',           # drop
    'Paris SG': 'PSG',
    'Pau': 'Pau',                   # drop
    'Pescara': 'Pescara',           # drop
    'Portsmouth': 'Portsmouth',     # drop
    'Preston': 'Preston',           # drop
    
    # Bloque Q
    'QPR': 'QPR',                   # drop
    
    # Bloque R
    'RC Lens': 'RacingLens',
    'Red Star': 'Red Star',         # drop
    'Reggiana': 'Reggiana',         # drop
    'Rodez': 'Rodez',               # drop
    'Real Madrid': 'RealMadrid',
    
    # Bloque S
    'SC Freiburg': 'Freiburg',
    'Saint-Étienne': 'SaintEtienne',
    'Santander': 'Santander',       # drop
    'Schalke': 'Schalke04',
    'Sheffield United': 'SheffieldUtd',
    'Sheffield Weds': 'Sheffield Wednesday', # drop
    'Sociedad B': 'Real Sociedad B', # drop
    'St. Pauli': 'StPauli',
    'Stade Brestois': 'StadeBrestois29',
    'Stade Rennais': 'StadeRennais',
    'Stade de Reims': 'StadeReims',
    'Suedtirol': 'Suedtirol',       # drop
    'Sunderland': 'Sunderland',     # drop
    
    # Bloque T
    'Tenerife': 'Tenerife',         # drop
    'Tottenham': 'Spurs',
    'Tottenham Hotspur': 'Spurs',
    'Toulouse FC': 'Toulouse',
    
    # Bloque V
    'Verona': 'Hellas',
    'VfB Stuttgart': 'Stuttgart',
    'VfL Wolfsburg': 'Wolfsburg',
    
    # Bloque W
    'Werder': 'WerderBremen',
    'West Brom': 'WestBrom',
    'West Ham': 'WestHam',
    'West Ham United': 'WestHam',
    'Wolves': 'Wolves',
    'Wrexham': 'Wrexham',           # drop
    
    # Bloque Y
    'YokofamaFMarinos': 'YokohamaMarinos',
    
    # Bloque Z
    'Zaragoza': 'Zaragoza',         # drop
    
    # Sudamericanos (minúsculas)
    'athletico paranaense': 'AthParanaense',
    'bahia': 'Bahia',
    'banfield': 'Banfield',
    'belgrano': 'Belgrano',
    'boca juniors': 'BocaJuniors',
    'botafogo': 'Botafogo',
    'carabobo': 'Carabobo',
    'chapecoense': 'Chapecoense',
    'corinthians': 'Corinthians',
    'coritiba': 'Coritiba',
    'cruzeiro': 'Cruzeiro',
    'defensa y justicia': 'DefensaYJusticia',
    'deportivo lara': 'DepLara',
    'flamengo': 'Flamengo',
    'fluminense': 'Fluminense',
    'godoy cruz': 'GodoyCruz',
    'independiente': 'IndependienteAvellaneda',
    'internacional': 'InterPA',
    'juventude': 'Juventude',
    'metropolitanos': 'Metropolitanos',
    'monagas': 'Monagas',
    'nan': 'nan',
    'newell s old boys': 'NewellsOldBoys',
    'palmeiras': 'Palmeiras',
    'patronato': 'Patronato',
    'platense': 'Platense',
    'ponte preta': 'Ponte Preta',
    'puerto cabello': 'AcPtoCabello',
    'racing club': 'RacingAvellaneda',
    'rayo zuliano': 'Zulia',
    'river plate': 'RiverPlate',
    'rosario central': 'RosarioCentral',
    'san lorenzo': 'SanLorenzo',
    'santos': 'Santos',
    'tigre': 'Tigre',
    'trujillanos': 'Trujillanos',
    'universidad central': 'UCVFC',
    'vasco da gama': 'VascoDaGama',
    'zulia': 'Zulia',
    
    # Griegos (caracteres nativos)
    'Άρης': 'Aris',
    'Αστέρας': 'AsterasTripolis',
    'Ατρόμητος': 'Atromitos',
    'Βόλος': 'Volos',
    'Καλλιθέα': 'AthensKallithea',
    'Λάρισα': 'AEL',
    'Λαμία': 'Lamia',
    'Λεβαδειακός': 'Levadiakos',
    'ΟΦΗ': 'OFI',
    'Ολυμπιακός': 'Olympiacos',
    'ΠΑΟΚ': 'PAOK',
    'Παναθηναϊκός': 'Panathinaikos',
    'Παναιτωλικός': 'Panetolikos',
    'Πανσερραϊκός': 'Panserraikos'
}

# 2. Extraer equipos #drop (los que tienen el comentario '#drop')
#    Para esto, simplemente listamos manualmente los nombres que marcaste.
#    Esta lista está basada en tu archivo.
drop_teams = [
    'Albacete', 'Almería', 'Andorra CF', 'Annecy', 'Avellino',
    'Bari', 'Bastia', 'Birmingham', 'Blackburn', 'Braunschweig',
    'Bristol City', 'Burgos', 'Carrarese', 'Cartagena', 'Castellón',
    'Catanzaro', 'Cesena', 'Ceuta', 'Charlton', 'Cittadella',
    'Cosenza', 'Coventry', 'Córdoba', 'Derby', 'Dresden',
    'Dunkerque', 'Eldense', 'Elversberg', 'Entella', 'Ferrol',
    'Gijón', 'Grenoble', 'Hull', 'Juve Stabia', 'Karlsruhe',
    'Lautern', 'Laval', 'Le Mans', 'Leonesa', 'Magdeburg',
    'Mantova', 'Martigues', 'Middlesbrough', 'Millwall', 'Mirandes',
    'Modena', 'Münster', 'Nancy', 'Oxford', 'Padova',
    'Palermo', 'Pau', 'Pescara', 'Portsmouth', 'Preston',
    'QPR', 'Red Star', 'Reggiana', 'Rodez', 'Santander',
    'Sheffield Weds', 'Sociedad B', 'Suedtirol', 'Sunderland',
    'Tenerife', 'Wrexham', 'Zaragoza'
]

# 3. Filtrar df_elo_aug: eliminar equipos que estén en drop_teams
#    Primero, asegurarnos de que la columna 'team' existe y es string
if 'team' in df_elo_aug.columns:
    df_elo_aug['club'] = df_elo_aug['team'].astype(str).str.strip()
    # Contar antes
    before = len(df_elo_aug)
    # Filtrar
    df_elo_aug_filtrado = df_elo_aug[~df_elo_aug['team'].isin(drop_teams)].copy()
    after = len(df_elo_aug_filtrado)
    print(f"✅ Equipos eliminados de ELO Agosto: {before - after}")
    print(f"✅ Equipos restantes en ELO Agosto: {after}")
else:
    print("⚠️ No se encontró la columna 'team' en df_elo_aug.")

# 4. Aplicar el mapeo manual a los equipos que quedan (normalización)
df_elo_aug_filtrado['team'] = df_elo_aug_filtrado['team'].map(manual_mapping).fillna(df_elo_aug_filtrado['club'])

print("\n✅ Normalización y filtro aplicados a ELO Agosto.")

# 5. Verificar algunos equipos para confirmar
print("\n🔍 Muestra de equipos normalizados en ELO Agosto:")
print(df_elo_aug_filtrado[['team', 'country']].head(10))

In [ ]:
# =============================================
# VERIFICACIÓN DE EQUIPOS CONTRA MATCHES
# =============================================

import pandas as pd
import os

BASE_DATA_DIR = "/content/drive/MyDrive/TEG02-2025Arellano/Data"

# 1. Cargar el archivo de matches
matches_path = os.path.join(BASE_DATA_DIR, "matches_final_model_clean2.csv")
df_matches = pd.read_csv(matches_path)

print("📋 Archivo de matches cargado.")
print(f"   Shape: {df_matches.shape}")
print(f"   Columnas: {df_matches.columns.tolist()}")

# 2. Extraer equipos únicos de la columna 'home_team'
if 'home_team' in df_matches.columns:
    home_teams = set(df_matches['home_team'].astype(str).str.strip().unique())
    print(f"   Equipos únicos en 'home_team': {len(home_teams)}")
else:
    print("⚠️ No se encontró la columna 'home_team'.")
    home_teams = set()

# También podríamos incluir 'away_team' si existe
if 'away_team' in df_matches.columns:
    away_teams = set(df_matches['away_team'].astype(str).str.strip().unique())
    print(f"   Equipos únicos en 'away_team': {len(away_teams)}")
    # Unir ambos conjuntos
    all_match_teams = home_teams.union(away_teams)
else:
    all_match_teams = home_teams

print(f"   Total de equipos únicos en matches (home+away): {len(all_match_teams)}")

# 3. Verificar cada DataFrame
dataframes_dict = {
    'UEFA': df_uefa_filtrado,
    'AFC': df_afc_filtrado,
    'ELO_Ene': df_elo_jan_filtrado,
    'ELO_Ago': df_elo_aug_filtrado,
    'CONMEBOL': df_conmebol_filtrado
}

print("\n" + "="*60)
print("🔍 VERIFICACIÓN DE EQUIPOS CONTRA MATCHES")
print("="*60)

not_in_matches_summary = {}

for name, df in dataframes_dict.items():
    # Determinar la columna de equipo
    if 'team' in df.columns:
        team_col = 'team'
    elif 'club' in df.columns:
        team_col = 'club'
    else:
        print(f"⚠️ {name}: No tiene columna 'team' ni 'club'.")
        continue
    
    # Obtener equipos únicos en el DataFrame
    df_teams = set(df[team_col].astype(str).str.strip().unique())
    
    # Equipos que NO están en matches
    not_in_matches = df_teams - all_match_teams
    
    # Guardar resumen
    not_in_matches_summary[name] = sorted(not_in_matches)
    
    print(f"\n📌 {name}:")
    print(f"   Total equipos en DF: {len(df_teams)}")
    print(f"   Equipos que NO están en matches: {len(not_in_matches)}")
    if not_in_matches:
        print(f"   Muestra (primeros 20): {sorted(not_in_matches)[:20]}")
    else:
        print("   ✅ Todos los equipos están en matches.")

# 4. Exportar listas completas de los que no coinciden
for name, missing in not_in_matches_summary.items():
    if missing:
        output_path = os.path.join(BASE_DATA_DIR, f"equipos_no_en_matches_{name}.txt")
        with open(output_path, 'w') as f:
            for team in sorted(missing):
                f.write(team + '\n')
        print(f"\n📁 Lista de {name} guardada en: {output_path}")

print("\n✅ Verificación completada.")